# SignSage — Full iSign Sign→English Training Notebook — deadline final

This notebook trains the primary Sign→English model using the complete cleaned iSign v1.1
corpus: 123,779 usable rows after explicit data-quality filtering, with a leakage-free
video-level 80/10/10 train/validation/test split.

Important terminology: all 123,779 usable samples participate in the experiment; the model
is optimized on the training split, while validation/test remain held out for unbiased
model selection and evaluation. Do not describe held-out rows as training data.

The Kaggle pose archive remains remote and is streamed lazily by UID. The notebook uses
larger HTTP ranges, archive-local batches, persistent workers, and recovery checkpoints
to reduce remote I/O overhead.

The primary deliverable is a compact deployment bundle containing the checkpoint,
tokenizer, and exact preprocessing/model contract. Dataset files and caches are excluded.


## 0. Prerequisites

### Kaggle mode

The notebook is optimized for Kaggle GPU execution.

The large iSign pose archive is **not fully extracted into Kaggle storage**. Instead, the notebook uses the Hugging Face-backed multipart ZIP64 reader already implemented below and fetches only the pose members required by the selected samples.

Expected remote resources:

- `iSign_v1.1.csv`
- multipart iSign pose archive:
  - `iSign-poses_v1.1_part_aa`
  - `iSign-poses_v1.1_part_ab`
  - `iSign-poses_v1.1_part_ac`
  - `iSign-poses_v1.1_part_ad`
- Kaggle Secret: `HF_TOKEN`, with read access to `Exploration-Lab/iSign`

### Local mode

The same notebook can still run on the local storage machine when `SIGNSAGE_DATA_DIR` points to the iSign root directory containing:

```text
iSign/
├── iSign_v1.1.csv
└── poses/
    └── iSign-poses_v1.1/
        └── *.pose
```

### Execution order

Run the notebook **top to bottom**. The notebook is deliberately organized as:

```text
Environment
    ↓
Dataset verification
    ↓
Data audit
    ↓
Leak-safe video-level split
    ↓
Tokenizer / vocabulary
    ↓
Pose normalization + cache
    ↓
DataLoader smoke check
    ↓
Sign → Text models
    ↓
Sign → Text evaluation
    ↓
Text → Sign baselines / generator
    ↓
Text → Sign evaluation
    ↓
Visualization + round-trip
    ↓
Latency
    ↓
Knowledge repository finalization
    ↓
Research GO / NO-GO report
    ↓
Export manifest
```


## 0a. Performance & memory strategy

This is the FULL-CORPUS run. All 123,779 cleaned iSign rows are assigned to the
leakage-free train/validation/test split.

Kaggle does not need the entire pose archive locally. The four multipart ZIP ranges
are exposed as one seekable ZIP64 stream and individual UID-level `.pose` members are
fetched lazily.

Runtime safeguards/optimizations:
- Larger HTTP chunks reduce request/redirect overhead.
- Each DataLoader worker owns its own ZIP reader.
- Only two workers are used on Kaggle and they are non-persistent to prevent host-RAM accumulation.
- Archive-local batches reduce random HTTP range churn.
- Batch order is shuffled while preserving within-batch locality.
- Small per-worker chunk caches limit host RAM.
- Recovery checkpoints are written during training.
- Full-data mode does NOT abort merely because the remote path is slower than a nominal
  target; throughput is reported and training proceeds.
- Optional GRU, no-face ablation, and Text→Pose branches remain OFF to reserve compute
  for the primary model.

If the remote source itself becomes unavailable, the notebook fails loudly rather than
silently training on incomplete data.


In [ ]:
# SignSage configuration + environment detection
import os, gc, io, json, random, time, warnings, zipfile, math
from pathlib import Path
from collections import Counter
from functools import lru_cache
from contextlib import nullcontext

from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda": torch.cuda.manual_seed_all(SEED)

IS_KAGGLE = Path("/kaggle").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
DATA_MODE = "kaggle" if IS_KAGGLE else "local"

# FULL-CORPUS PRIMARY RUN.
# All cleaned iSign rows are assigned to train/validation/test.
# Only the training split receives gradient updates; validation/test remain held out.
RUN_MODE = "full" if IS_KAGGLE else "smoke"
assert RUN_MODE in {"full", "smoke"}

RUN_PRIMARY_S2T = True
RUN_GRU_BASELINE = False
RUN_NOFACE_ABLATION = False
RUN_TEXT_TO_POSE = False

if RUN_MODE == "smoke":
    TRAIN_LIMIT, VAL_LIMIT, TEST_LIMIT = 300, 60, 60
    N_EPOCHS_TRANSFORMER = 1
else:
    TRAIN_LIMIT = VAL_LIMIT = TEST_LIMIT = None
    # Two complete passes over the training split. Every training row is visited
    # once per epoch; validation/test are evaluated separately.
    N_EPOCHS_TRANSFORMER = 2

N_EPOCHS_GRU = 0
N_EPOCHS_POSE = 0

MAX_TOKENS = 96
TARGET_VOCAB_SIZE = 16000
MAX_FRAMES = 160
POSE_OUTPUT_FRAMES = 60

# Larger physical batches reduce per-batch Python/network overhead.
BATCH_SIZE = 64 if DEVICE.type == "cuda" else 4
NUM_WORKERS = 2 if IS_KAGGLE else 0
CACHE_SIZE = 256

REMOTE_BATCH_LOCALITY = True
REMOTE_CHUNK_SIZE_MB = 16
REMOTE_CACHE_CHUNKS = 4
PERF_PREFLIGHT_BATCHES = 12
# Informational only. The old hard stop is intentionally removed because it wasted
# the Kaggle session after measuring the existing 5.63 s/batch remote path.
PERF_MAX_SECONDS_PER_BATCH = 5.0

GRAD_ACCUM_STEPS = 1 if DEVICE.type == "cuda" else 4
USE_AMP = DEVICE.type == "cuda"
USE_BF16_AUTOCAST_CPU = False

# Do not build a multi-GB normalized-pose cache on Kaggle.
USE_POSE_DISK_CACHE = False
PREWARM_CACHE = False

USE_LLM_GLOSS = False
DTW_EVAL_N = 200
RETRIEVAL_MAX_SAMPLES = 3000

BEAM_WIDTH = 4
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 3
MIN_OUTPUT_TOKENS = 1
MAX_DECODE_TOKENS = MAX_TOKENS - 2

def normalize_text(text):
    return " ".join(str(text).lower().strip().split())

def autocast_ctx():
    if USE_AMP: return torch.autocast(device_type="cuda", dtype=torch.float16)
    if USE_BF16_AUTOCAST_CPU and DEVICE.type == "cpu":
        try: return torch.autocast(device_type="cpu", dtype=torch.bfloat16)
        except Exception: pass
    return nullcontext()

try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def check_memory():
    try:
        import psutil
        mem, swap = psutil.virtual_memory(), psutil.swap_memory()
        print(f"RAM: {mem.used/1024**3:.1f} / {mem.total/1024**3:.1f} GB ({mem.percent}%)")
        print(f"Swap: {swap.used/1024**3:.1f} / {swap.total/1024**3:.1f} GB ({swap.percent}%)")
        return mem, swap
    except Exception:
        return None, None

def maybe_check_memory():
    try: check_memory()
    except Exception: pass

if DATA_MODE == "local":
    DATA_DIR = Path(os.environ.get("SIGNSAGE_DATA_DIR", "")).expanduser()
    if not str(DATA_DIR): raise RuntimeError("SIGNSAGE_DATA_DIR is not set.")
else:
    DATA_DIR = Path("/kaggle/working/signsage_data"); DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "iSign_v1.1.csv"
POSE_DIR = DATA_DIR / "poses" / "iSign-poses_v1.1"

OUTPUT_DIR = Path("/kaggle/working/signsage_outputs") if IS_KAGGLE else Path("signsage_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"; CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR = OUTPUT_DIR / "reports"; REPORT_DIR.mkdir(parents=True, exist_ok=True)
KNOWLEDGE_DB_PATH = OUTPUT_DIR / "signsage_knowledge.db"
KNOWLEDGE_JSON_PATH = OUTPUT_DIR / "signsage_knowledge_manifest.json"

RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
RUN_STARTED_AT = pd.Timestamp.utcnow().isoformat()

print("Device:", DEVICE, "| Data mode:", DATA_MODE, "| Run mode:", RUN_MODE)
print("Primary Sign→Text:", RUN_PRIMARY_S2T)
print("All cleaned iSign rows across split:", "YES" if RUN_MODE == "full" else "NO")
print("AMP:", USE_AMP, "| Disk pose cache:", USE_POSE_DISK_CACHE)
if RUN_MODE == "full":
    print("FULL-CORPUS RUN: all cleaned rows will be assigned to train/validation/test.")
else:
    print("SMOKE RUN: bounded execution outside Kaggle.")


In [ ]:
# 0b. Project knowledge repository (SQLite + JSON)
#
# Kaggle is the experiment workspace. This repository is the structured record
# that will later be copied into the main SignSage project for backend use.
#
# SQLite = queryable historical/research record
# JSON    = portable run manifest / configuration snapshot

import sqlite3
from pathlib import Path
from datetime import datetime, timezone

OUTPUT_DIR = Path("/kaggle/working/signsage_outputs") if DATA_MODE == "kaggle" else Path.cwd() / "signsage_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = OUTPUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

KNOWLEDGE_DB_PATH = OUTPUT_DIR / "signsage_knowledge.db"
KNOWLEDGE_JSON_PATH = OUTPUT_DIR / "signsage_knowledge_manifest.json"

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

def _sql_connect():
    conn = sqlite3.connect(KNOWLEDGE_DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def init_knowledge_repository():
    conn = _sql_connect()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS runs (
        run_id TEXT PRIMARY KEY,
        started_at TEXT NOT NULL,
        completed_at TEXT,
        data_mode TEXT NOT NULL,
        run_mode TEXT NOT NULL,
        device TEXT NOT NULL,
        gpu_count INTEGER NOT NULL,
        seed INTEGER NOT NULL
    );

    CREATE TABLE IF NOT EXISTS dataset (
        run_id TEXT PRIMARY KEY,
        dataset_name TEXT NOT NULL,
        csv_path TEXT,
        csv_rows INTEGER,
        unique_video_ids INTEGER,
        pose_source TEXT,
        selected_rows INTEGER,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS splits (
        run_id TEXT PRIMARY KEY,
        train_rows INTEGER,
        val_rows INTEGER,
        test_rows INTEGER,
        train_videos INTEGER,
        val_videos INTEGER,
        test_videos INTEGER,
        leakage_free INTEGER,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS experiments (
        experiment_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        name TEXT NOT NULL,
        family TEXT NOT NULL,
        input_features TEXT,
        epochs INTEGER,
        batch_size INTEGER,
        learning_rate REAL,
        weight_decay REAL,
        checkpoint_path TEXT,
        best_validation_loss REAL,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS metrics (
        metric_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        experiment_name TEXT NOT NULL,
        metric_name TEXT NOT NULL,
        metric_value REAL,
        split_name TEXT,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS artifacts (
        artifact_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        artifact_name TEXT NOT NULL,
        artifact_type TEXT NOT NULL,
        path TEXT NOT NULL,
        description TEXT,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );

    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        run_id TEXT NOT NULL,
        key TEXT NOT NULL,
        value TEXT,
        FOREIGN KEY(run_id) REFERENCES runs(run_id)
    );
    """)
    conn.execute(
        """INSERT OR REPLACE INTO runs
           (run_id, started_at, completed_at, data_mode, run_mode, device, gpu_count, seed)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
        (RUN_ID, RUN_STARTED_AT, None, DATA_MODE, RUN_MODE, str(DEVICE),
         int(torch.cuda.device_count()), int(SEED)),
    )
    conn.commit()
    conn.close()

def knowledge_insert(table, columns, values, delete_keys=None):
    """Insert one row, optionally deleting any existing row(s) that match
    `delete_keys` first. Tables whose primary key is `run_id` (dataset,
    splits) don't need this -- INSERT OR REPLACE already keys off it -- but
    tables with an autoincrement id (experiments) have no natural
    uniqueness, so without an explicit delete, re-running a training cell
    silently appends a duplicate row instead of replacing it."""
    conn = _sql_connect()
    if delete_keys:
        where_clause = " AND ".join(f"{col} IS ?" for col in delete_keys)
        conn.execute(f"DELETE FROM {table} WHERE {where_clause}", tuple(delete_keys.values()))
    placeholders = ", ".join(["?"] * len(values))
    conn.execute(
        f"INSERT OR REPLACE INTO {table} ({', '.join(columns)}) VALUES ({placeholders})",
        tuple(values),
    )
    conn.commit()
    conn.close()

def knowledge_metric(experiment_name, metric_name, value, split_name=None):
    conn = _sql_connect()
    # Idempotent: replace the matching (run, experiment, metric, split) row
    # instead of appending a duplicate on re-run. `IS ?` (not `= ?`) so this
    # also matches correctly when split_name is NULL.
    conn.execute(
        """DELETE FROM metrics
           WHERE run_id = ? AND experiment_name = ? AND metric_name = ? AND split_name IS ?""",
        (RUN_ID, experiment_name, metric_name, split_name),
    )
    conn.execute(
        """INSERT INTO metrics
           (run_id, experiment_name, metric_name, metric_value, split_name)
           VALUES (?, ?, ?, ?, ?)""",
        (RUN_ID, experiment_name, metric_name, float(value), split_name),
    )
    conn.commit()
    conn.close()

def knowledge_artifact(name, artifact_type, path, description=""):
    conn = _sql_connect()
    # Idempotent: one row per (run_id, artifact_name); re-running a cell
    # updates the existing artifact record instead of duplicating it.
    conn.execute(
        "DELETE FROM artifacts WHERE run_id = ? AND artifact_name = ?",
        (RUN_ID, name),
    )
    conn.execute(
        """INSERT INTO artifacts
           (run_id, artifact_name, artifact_type, path, description)
           VALUES (?, ?, ?, ?, ?)""",
        (RUN_ID, name, artifact_type, str(path), description),
    )
    conn.commit()
    conn.close()

def knowledge_observation(key, value):
    # Deliberately append-only (unlike the writers above): observations such
    # as "go_no_go" are log-style notes, not a single fact per key, so a
    # re-run recording another entry is the correct behavior here.
    conn = _sql_connect()
    conn.execute(
        "INSERT INTO observations (run_id, key, value) VALUES (?, ?, ?)",
        (RUN_ID, str(key), json.dumps(value, default=str)),
    )
    conn.commit()
    conn.close()

init_knowledge_repository()

print("Knowledge DB :", KNOWLEDGE_DB_PATH)
print("Knowledge JSON:", KNOWLEDGE_JSON_PATH)
print("Run ID       :", RUN_ID)


In [ ]:
# Kaggle-only dependency installation.
# Kaggle's default GPU image does not include pose-format, jiwer, sacrebleu,
# or rouge-score, and its bundled huggingface_hub can be too old for
# `hf_hub_download(..., local_dir=...)` to behave reliably. Installing these
# here -- before anything below imports them -- is what actually lets the
# rest of this notebook run top-to-bottom on a fresh Kaggle kernel; without
# it, the very next cell's `from pose_format import Pose` raises
# ModuleNotFoundError before the dataset is even touched.
#
# Local-machine venvs are expected to already have these installed (see
# Section 0 / the commented-out install line in the original local notebook),
# so this deliberately does nothing outside Kaggle mode.
if IS_KAGGLE:
    import subprocess
    import sys as _sys

    _KAGGLE_ONLY_PACKAGES = [
        "pose-format",
        "jiwer",
        "sacrebleu",
        "rouge-score",
        "sentencepiece",
        "huggingface_hub>=0.24.0",
        "requests",  # used directly by the remote ZIP64 backend (cell below);
                     # declared explicitly rather than assumed from the base image.
    ]
    print("Kaggle mode: installing packages not present in the default image:")
    print(" ", ", ".join(_KAGGLE_ONLY_PACKAGES))
    subprocess.run(
        [_sys.executable, "-m", "pip", "install", "-q", "--no-input", *_KAGGLE_ONLY_PACKAGES],
        check=True,
    )
    print("Done. (Safe to re-run this cell; pip install is idempotent.)")
else:
    print("Local mode: skipping Kaggle-only package installation "
          "(the local venv is expected to already have these -- see Section 0).")


In [ ]:
# 1. Verify the iSign dataset before training.
if "DATA_MODE" not in globals():
    raise RuntimeError("Run the configuration cell first.")

if DATA_MODE == "local":
    assert CSV_PATH.is_file(), f"Missing CSV: {CSV_PATH}"
    assert POSE_DIR.is_dir(), f"Missing pose directory: {POSE_DIR}"
    pose_files = sorted(POSE_DIR.glob("*.pose"))
else:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download
    try:
        HF_TOKEN = os.environ.get("HF_TOKEN") or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("Add HF_TOKEN as a Kaggle Secret.") from exc
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN is empty.")
    CSV_PATH = Path(hf_hub_download(
        repo_id="Exploration-Lab/iSign", filename="iSign_v1.1.csv",
        repo_type="dataset", token=HF_TOKEN, local_dir=str(DATA_DIR)
    ))
    pose_files = []

df = pd.read_csv(CSV_PATH)
RAW_CSV_ROWS = len(df)
required = {"uid", "text"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

df["uid"] = df["uid"].astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df[["video_id", "seq"]] = df["uid"].str.rsplit("-", n=1, expand=True)
# A valid split key is mandatory: rows whose UID cannot be mapped to a video
# must never silently disappear from every split.
bad_video_id = df["video_id"].isna() | (df["video_id"].astype(str).str.strip() == "")
bad_seq = df["seq"].isna() | (df["seq"].astype(str).str.strip() == "")
malformed_uid = int((bad_video_id | bad_seq).sum())
if malformed_uid:
    print(f"Removing {malformed_uid:,} rows with malformed UID/video/sequence fields.")
    df = df.loc[~(bad_video_id | bad_seq)].copy()
df["video_id"] = df["video_id"].astype(str).str.strip()
df["seq"] = df["seq"].astype(str).str.strip()
df["n_words"] = df["text"].str.split().str.len()

# Remove only demonstrably non-sentence/caption artifacts. Legitimate one-word
# English targets (for example "hello" or "thanks") are retained.
before = len(df)
text_norm = df["text"].map(normalize_text)
known_junk = (
    text_norm.eq("")
    | text_norm.str.fullmatch(r"\d+")
    | text_norm.str.fullmatch(r"[a-z]")
    | text_norm.str.fullmatch(r"\[[^\]]+\]")
    | text_norm.str.fullmatch(r"[^a-z0-9]+")
)
# "i" is a legitimate English word, so the single-letter rule above must not
# remove it. Keep the row if it is exactly the pronoun "i".
known_junk &= ~text_norm.eq("i")
known_junk |= df["uid"].eq("")
junk_counts = {
    "empty_text": int(text_norm.eq("").sum()),
    "numeric_only": int(text_norm.str.fullmatch(r"\d+").sum()),
    "single_letter_non_i": int((text_norm.str.fullmatch(r"[a-z]") & ~text_norm.eq("i")).sum()),
    "bracketed_caption": int(text_norm.str.fullmatch(r"\[[^\]]+\]").sum()),
    "punctuation_only": int(text_norm.str.fullmatch(r"[^a-z0-9]+").sum()),
}
df = df.loc[~known_junk].copy()
after = len(df)

duplicate_uid = int(df["uid"].duplicated().sum())
if duplicate_uid:
    raise RuntimeError(f"Found {duplicate_uid} duplicate UIDs after cleaning; refusing to train until resolved.")

# Report long targets instead of silently truncating them later.
too_long = int((df["n_words"] > MAX_TOKENS - 2).sum())
print(f"Raw CSV rows: {RAW_CSV_ROWS:,}")
print(f"Usable non-empty rows: {after:,}")
print(f"Rows removed by text/caption cleaning: {before-after:,}")
print(f"Malformed UID rows removed: {malformed_uid:,}")
print(f"Known junk/caption rows removed (overlapping categories may be counted once in total): {before-after:,}")
print("Junk categories:", junk_counts)
print(f"Rows longer than {MAX_TOKENS-2} words: {too_long:,}")
print(f"Unique video IDs: {df['video_id'].nunique():,}")

if DATA_MODE == "local":
    print(f"Local pose files: {len(pose_files):,}")
else:
    print("Pose source: remote multipart ZIP64; individual UID members will be streamed.")

assert len(df) > 0


In [ ]:
# 1a. Record dataset facts in the knowledge repository.
pose_source = (
    "local UID-level .pose files"
    if DATA_MODE == "local"
    else "remote multipart ZIP64 archive on Hugging Face; lazy member reads"
)
selected_rows = len(df)

knowledge_insert(
    "dataset",
    ["run_id", "dataset_name", "csv_path", "csv_rows", "unique_video_ids", "pose_source", "selected_rows"],
    [RUN_ID, "iSign v1.1", str(CSV_PATH), int(len(df)), int(df["video_id"].nunique()), pose_source, int(selected_rows)],
)
knowledge_observation("dataset_columns", df.columns.tolist())
knowledge_observation("text_empty_rows", int((df["text"].str.strip() == "").sum()))

print("Dataset metadata recorded in SQLite.")


In [ ]:
# Kaggle remote ZIP64 backend
# The four iSign pose files are consecutive byte ranges of ONE ZIP archive.
# This backend presents them as one seekable virtual file so Python zipfile can
# read the ZIP64 central directory and then fetch only requested members.
# Reads are explicitly bounded to both part and HTTP-chunk boundaries so no bytes
# are skipped when zipfile asks for a range that crosses a cached chunk.

if DATA_MODE == "kaggle":
    import re
    import time
    import requests
    from huggingface_hub import hf_hub_url

    REMOTE_PARTS = [
        "iSign-poses_v1.1_part_aa",
        "iSign-poses_v1.1_part_ab",
        "iSign-poses_v1.1_part_ac",
        "iSign-poses_v1.1_part_ad",
    ]
    REMOTE_CHUNK_SIZE = REMOTE_CHUNK_SIZE_MB * 1024 * 1024
    REMOTE_CONNECT_TIMEOUT = 20
    REMOTE_READ_TIMEOUT = 180
    REMOTE_RETRIES = 5
    REMOTE_BACKOFF_SECONDS = 1.5

    class RemoteConcatFile(io.RawIOBase):
        def __init__(self, urls, sizes, token, chunk_size=REMOTE_CHUNK_SIZE):
            super().__init__()
            self.urls = urls
            self.sizes = sizes
            self.starts = np.cumsum([0] + sizes[:-1]).tolist()
            self.total_size = int(sum(sizes))
            self.pos = 0
            self.token = token
            self.chunk_size = int(chunk_size)
            self.session = requests.Session()
            self.session.headers.update({
                "Authorization": f"Bearer {token}",
                "User-Agent": "SignSage-Kaggle/1.0",
            })
            from collections import OrderedDict
            self.cache = OrderedDict()
            # Caches the final (post-redirect) CDN URL per part so repeated chunk
            # fetches skip the huggingface.co auth+redirect hop. Cleared and
            # re-resolved automatically if a cached link ever expires (see _fetch).
            self.resolved = {}

        def readable(self):
            return True

        def seekable(self):
            return True

        def tell(self):
            return self.pos

        def seek(self, offset, whence=os.SEEK_SET):
            if whence == os.SEEK_SET:
                new = offset
            elif whence == os.SEEK_CUR:
                new = self.pos + offset
            elif whence == os.SEEK_END:
                new = self.total_size + offset
            else:
                raise ValueError("invalid whence")
            if new < 0:
                raise ValueError("negative seek position")
            self.pos = min(int(new), self.total_size)
            return self.pos

        def _part_for(self, pos):
            idx = int(np.searchsorted(self.starts, pos, side="right") - 1)
            idx = max(0, min(idx, len(self.urls) - 1))
            return idx, pos - self.starts[idx]

        def _fetch(self, part_idx, local_start, n):
            key = (part_idx, (local_start // self.chunk_size) * self.chunk_size)
            if key in self.cache:
                self.cache.move_to_end(key)
                returned_start, content = self.cache[key]
                offset = local_start - returned_start
                piece = content[offset:offset + n]
                if len(piece) == n:
                    return piece
                # A corrupted/incomplete cache entry must never be served.
                self.cache.pop(key, None)

            chunk_start = key[1]
            chunk_end = min(
                self.sizes[part_idx] - 1,
                chunk_start + self.chunk_size - 1,
            )
            headers = {"Range": f"bytes={chunk_start}-{chunk_end}"}

            last_error = None
            for attempt in range(1, REMOTE_RETRIES + 1):
                try:
                    # Resolve the origin URL once, then reuse the CDN URL.
                    request_url = self.resolved.get(part_idx, self.urls[part_idx])
                    r = self.session.get(
                        request_url,
                        headers=headers,
                        timeout=(REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT),
                        allow_redirects=True,
                        stream=False,
                    )

                    if r.status_code != 206 and request_url != self.urls[part_idx]:
                        # Cached CDN URL may have expired. Re-resolve once.
                        self.resolved.pop(part_idx, None)
                        r.close()
                        r = self.session.get(
                            self.urls[part_idx],
                            headers=headers,
                            timeout=(REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT),
                            allow_redirects=True,
                            stream=False,
                        )

                    if r.status_code != 206:
                        raise RuntimeError(
                            f"Range request failed for part {part_idx}: HTTP {r.status_code}"
                        )

                    cr = r.headers.get("Content-Range", "")
                    match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", cr.strip())
                    if not match:
                        raise RuntimeError(
                            f"Invalid Content-Range from HF response: {cr!r}"
                        )

                    returned_start, returned_end, returned_total = map(int, match.groups())
                    if returned_total != self.sizes[part_idx]:
                        raise RuntimeError(
                            f"HF reported part {part_idx} total size {returned_total}, "
                            f"but the size probe reported {self.sizes[part_idx]}."
                        )
                    if not (returned_start <= chunk_start and chunk_end <= returned_end):
                        raise RuntimeError(
                            f"HF returned byte range {returned_start}-{returned_end}; "
                            f"requested {chunk_start}-{chunk_end}."
                        )

                    content = r.content
                    expected_response_len = returned_end - returned_start + 1
                    if len(content) != expected_response_len:
                        raise RuntimeError(
                            f"Incomplete range response for part {part_idx}: "
                            f"Content-Range says {expected_response_len} bytes, "
                            f"but received {len(content)}."
                        )

                    self.resolved[part_idx] = r.url
                    self.cache[key] = (returned_start, content)
                    self.cache.move_to_end(key)
                    while len(self.cache) > REMOTE_CACHE_CHUNKS:
                        self.cache.popitem(last=False)

                    offset = local_start - returned_start
                    piece = content[offset:offset + n]
                    if len(piece) != n:
                        raise RuntimeError(
                            f"Remote range backend returned {len(piece)} bytes; expected {n}."
                        )
                    return piece

                except (requests.exceptions.Timeout,
                        requests.exceptions.ConnectionError,
                        requests.exceptions.ChunkedEncodingError) as exc:
                    last_error = exc
                    self.resolved.pop(part_idx, None)
                    try:
                        r.close()
                    except Exception:
                        pass
                    if attempt < REMOTE_RETRIES:
                        delay = REMOTE_BACKOFF_SECONDS * (2 ** (attempt - 1))
                        print(
                            f"[iSign remote] transient network error on part {part_idx}, "
                            f"range {chunk_start}-{chunk_end}; retry "
                            f"{attempt}/{REMOTE_RETRIES} after {delay:.1f}s"
                        )
                        time.sleep(delay)
                    continue
                except Exception:
                    try:
                        r.close()
                    except Exception:
                        pass
                    raise

            raise ConnectionError(
                f"Unable to fetch iSign ZIP range {chunk_start}-{chunk_end} "
                f"from part {part_idx} after {REMOTE_RETRIES} attempts."
            ) from last_error

        def read(self, n=-1):
            if self.pos >= self.total_size:
                return b""
            if n is None or n < 0:
                n = self.total_size - self.pos
            n = min(int(n), self.total_size - self.pos)
            out = bytearray()
            remaining = n
            while remaining:
                part_idx, local = self._part_for(self.pos)
                bytes_to_part_end = self.sizes[part_idx] - local
                bytes_to_chunk_end = self.chunk_size - (local % self.chunk_size)
                take = min(remaining, bytes_to_part_end, bytes_to_chunk_end)
                piece = self._fetch(part_idx, local, take)
                out.extend(piece)
                self.pos += take
                remaining -= take
            return bytes(out)

    def _remote_size(url):
        # A one-byte Range request gives the authoritative total size via Content-Range
        # without downloading a huge file.
        r = requests.get(
            url,
            headers={
                "Authorization": f"Bearer {HF_TOKEN}",
                "Range": "bytes=0-0",
                "User-Agent": "SignSage-Kaggle/1.0",
            },
            timeout=(REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT),
            allow_redirects=True,
        )
        if r.status_code != 206:
            raise RuntimeError(f"Could not probe remote part size: HTTP {r.status_code}")
        cr = r.headers.get("Content-Range", "")
        total = cr.rsplit("/", 1)[-1] if "/" in cr else ""
        if not total.isdigit():
            raise RuntimeError(f"Could not parse total size from Content-Range: {cr!r}")
        return int(total)

    remote_urls = [
        hf_hub_url("Exploration-Lab/iSign", f, repo_type="dataset")
        for f in REMOTE_PARTS
    ]
    remote_sizes = []
    for name, url in zip(REMOTE_PARTS, remote_urls):
        size = _remote_size(url)
        remote_sizes.append(size)
        print(f"{name}: {size / 1024**3:.2f} GiB")

    remote_file = RemoteConcatFile(remote_urls, remote_sizes, HF_TOKEN)
    remote_zip = zipfile.ZipFile(remote_file, mode="r", allowZip64=True)
    remote_names = set(remote_zip.namelist())
    print(f"Remote ZIP opened: {len(remote_names):,} members")

    # The member-path layout inside this archive is an assumption, not a verified
    # fact: RemotePoseStore below only tries "iSign-poses_v1.1/{uid}.pose" and
    # "{uid}.pose". Printing a few real member names here means a different prefix
    # or nesting depth is visible immediately, on the very first Kaggle run --
    # instead of surfacing later as a FileNotFoundError on every single UID, after
    # the split/tokenizer/cache cells have already run.
    print("Sample members:", sorted(remote_names)[:5])

    class RemotePoseStore:
        def __init__(self, zf, names):
            self.zf = zf
            self.names = names

        def _candidate(self, uid):
            candidates = (
                f"iSign-poses_v1.1/{uid}.pose",
                f"{uid}.pose",
            )
            for name in candidates:
                if name in self.names:
                    return name
            raise FileNotFoundError(
                f"UID not found in remote iSign pose archive: {uid}"
            )

        def info_for_uid(self, uid):
            """Return ZIP metadata without downloading the pose payload."""
            return self.zf.getinfo(self._candidate(uid))

        def read_member_bytes(self, uid):
            with self.zf.open(self._candidate(uid), "r") as f:
                return f.read()

    remote_pose_store = RemotePoseStore(remote_zip, remote_names)
    print("Remote pose backend ready — individual .pose members will be fetched lazily.")

    # Verify that every CSV UID has a corresponding .pose member using the ZIP
    # central directory only. This does NOT download all pose files.
    def _find_remote_member_name(uid):
        for candidate in (f"iSign-poses_v1.1/{uid}.pose", f"{uid}.pose"):
            if candidate in remote_names:
                return candidate
        return None

    _remote_uid_missing = []
    for uid in df["uid"].astype(str):
        if _find_remote_member_name(uid) is None:
            _remote_uid_missing.append(uid)
    print(f"Remote pose coverage: {len(df)-len(_remote_uid_missing):,}/{len(df):,} CSV UIDs")
    if _remote_uid_missing:
        print("First missing UIDs:", _remote_uid_missing[:10])
        raise RuntimeError(
            f"{len(_remote_uid_missing):,} CSV UIDs have no matching .pose member. "
            "Refusing to start full training with incomplete pose coverage."
        )
else:
    remote_pose_store = None

# Backward-compatible alias for any legacy helper; new network code uses the explicit connect/read timeouts above.
REMOTE_TIMEOUT = (REMOTE_CONNECT_TIMEOUT, REMOTE_READ_TIMEOUT)


## 2. Data audit

The audit is deliberately before training. A model cannot repair systematic text/pose misalignment.

The local dataset inspection already established that the `.pose` archive uses **row/UID-level files**, not `<video_id>.pose`. Therefore every CSV row is resolved through its complete UID.

In [ ]:
from pose_format import Pose

_worker_remote_pose_store = None

def _get_active_remote_pose_store():
    """Return a process-local ZIP reader; never share a seekable ZIP across workers."""
    global _worker_remote_pose_store
    if DATA_MODE != "kaggle":
        return remote_pose_store
    from torch.utils.data import get_worker_info
    if get_worker_info() is None:
        return remote_pose_store
    if _worker_remote_pose_store is None:
        assert REMOTE_CHUNK_SIZE > 0
        worker_file = RemoteConcatFile(remote_urls, remote_sizes, HF_TOKEN, REMOTE_CHUNK_SIZE)
        worker_zip = zipfile.ZipFile(worker_file, mode="r", allowZip64=True)
        _worker_remote_pose_store = RemotePoseStore(worker_zip, set(worker_zip.namelist()))
    return _worker_remote_pose_store

def read_pose_bytes_for_uid(uid):
    if DATA_MODE == "local":
        path = POSE_DIR / f"{uid}.pose"
        if not path.is_file():
            raise FileNotFoundError(path)
        return path.read_bytes()
    return _get_active_remote_pose_store().read_member_bytes(uid)

# Inspect a few actual pose files through the active backend.
sample_uids = df.head(5)["uid"].tolist()
for uid in sample_uids:
    p = Pose.read(read_pose_bytes_for_uid(uid))
    data = np.asarray(np.ma.filled(p.body.data, 0.0))
    conf = np.asarray(np.ma.filled(p.body.confidence, 0.0))
    print(f"{uid}.pose: shape={data.shape}, mean_confidence={conf.mean():.3f}")

example_pose = Pose.read(read_pose_bytes_for_uid(sample_uids[0]))
example_data = np.asarray(np.ma.filled(example_pose.body.data, 0.0))
TOTAL_KEYPOINTS = example_data.shape[2]
print("\nVerified pose shape:", example_data.shape)
print("Keypoints:", TOTAL_KEYPOINTS)


In [ ]:
## 2. Data integrity audit before training

# Dataset-use policy:
# - Primary training/evaluation: complete cleaned iSign v1.1 table and matching UID-level .pose members.
# - Excluded: only empty text and demonstrable non-sentence/caption artifacts.
# - Retained: legitimate one-word English targets.
# - INCLUDE is not merged into Sign→English training because it is an isolated-sign
#   dataset with a different task structure; it remains a separate reference/baseline dataset.

# These checks are intentionally cheap (CSV-only) and run over the entire usable iSign table.
# They answer whether the "large dataset" is actually being used and whether labels are
# duplicated/conflicting before any GPU time is spent.
print("=== FULL DATA AUDIT ===")
print("Rows available after non-empty-text cleaning:", f"{len(df):,}")
print("Unique UIDs:", f"{df.uid.nunique():,}")
print("Unique videos:", f"{df.video_id.nunique():,}")
print("Unique normalized English targets:", f"{df.text.map(normalize_text).nunique():,}")
print("Rows with 1 word:", int((df.n_words==1).sum()))
print("Rows > max target word length:", int((df.n_words>MAX_TOKENS-2).sum()))
spm_estimated_truncation_note = "SentencePiece token length is audited after tokenizer training."

# Duplicate UID must never map to multiple pose files.
assert df.uid.is_unique, "UID duplication detected."

# Report repeated target sentences; repetition is not itself an error.
target_counts=df.text.map(normalize_text).value_counts()
print("Most frequent targets:")
for text,count in target_counts.head(10).items():
    print(f"  {count:5d} × {text}")

# Check for exact duplicate UID/text rows before split.
dupes=int(df.duplicated(subset=["uid","text"]).sum())
print("Exact duplicate UID/text rows:",dupes)

knowledge_observation("full_dataset_audit",{
    "usable_rows":int(len(df)),
    "unique_uids":int(df.uid.nunique()),
    "unique_videos":int(df.video_id.nunique()),
    "unique_targets":int(df.text.map(normalize_text).nunique()),
    "one_word_rows":int((df.n_words==1).sum()),
    "too_long_rows":int((df.n_words>MAX_TOKENS-2).sum()),
    "exact_uid_text_duplicates":dupes,
})


# Full-run coverage statement: this is a row-level experiment over every cleaned
# CSV UID in the selected split; pose members are streamed lazily rather than
# downloaded/extracted as a monolithic archive.
print("Training policy: every row in the selected training split is eligible and visited once per epoch.")
if RUN_MODE != "full":
    print("This run is bounded; results must not be described as full-dataset training.")


In [ ]:
# 3. Leak-safe split by video_id.
video_ids = np.array(df["video_id"].dropna().unique(), dtype=object)
rng = np.random.default_rng(SEED); rng.shuffle(video_ids)

n_videos = len(video_ids)
train_cut, val_cut = int(.80*n_videos), int(.90*n_videos)
train_ids = set(video_ids[:train_cut]); val_ids = set(video_ids[train_cut:val_cut]); test_ids = set(video_ids[val_cut:])

df_train_all = df[df.video_id.isin(train_ids)].reset_index(drop=True)
df_val_all = df[df.video_id.isin(val_ids)].reset_index(drop=True)
df_test_all = df[df.video_id.isin(test_ids)].reset_index(drop=True)

# Do not drop long but otherwise valid samples. The tokenizer truncates targets only
# when they exceed MAX_TOKENS-2, and the truncation count is reported in the audit.
# This keeps every non-empty iSign row represented in the split.
df_train_all = df_train_all.reset_index(drop=True)
df_val_all = df_val_all.reset_index(drop=True)
df_test_all = df_test_all.reset_index(drop=True)

def limit_frame(frame, limit):
    if limit is None or limit >= len(frame):
        return frame.copy().reset_index(drop=True)
    return frame.sample(int(limit), random_state=SEED).reset_index(drop=True)

df_train = limit_frame(df_train_all, TRAIN_LIMIT)
df_val = limit_frame(df_val_all, VAL_LIMIT)
df_test = limit_frame(df_test_all, TEST_LIMIT)

print(f"Usable split rows: train={len(df_train_all):,}, val={len(df_val_all):,}, test={len(df_test_all):,}")
print(f"Rows used this run: train={len(df_train):,}, val={len(df_val):,}, test={len(df_test):,}")
# The *_all frames must account for every cleaned row. The *_run frames may
# be bounded in smoke/scaled modes; in full mode they must equal *_all.
assert len(df_train_all) + len(df_val_all) + len(df_test_all) == len(df), (
    "Leak-safe full split does not account for every cleaned row."
)
assert len(df_train) <= len(df_train_all) and len(df_val) <= len(df_val_all) and len(df_test) <= len(df_test_all)
if TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None:
    assert len(df_train) == len(df_train_all) and len(df_val) == len(df_val_all) and len(df_test) == len(df_test_all)
print(f"All cleaned rows assigned to exactly one full split: {len(df_train_all)+len(df_val_all)+len(df_test_all):,}/{len(df):,}")
assert set(df_train.video_id).isdisjoint(df_val.video_id)
assert set(df_train.video_id).isdisjoint(df_test.video_id)
assert set(df_val.video_id).isdisjoint(df_test.video_id)
print("No video_id leakage across train/val/test.")


In [ ]:
# 3b. Explicit dataset-usage report
cleaned_rows = len(df)
split_rows = len(df_train_all) + len(df_val_all) + len(df_test_all)
assert split_rows == cleaned_rows
print("=== DATA USED BY PRIMARY SIGN→ENGLISH EXPERIMENT ===")
print(f"Original CSV rows:       {RAW_CSV_ROWS:,}")
print(f"Cleaned usable rows:     {cleaned_rows:,}")
print(f"Rows excluded by cleaning: {RAW_CSV_ROWS-cleaned_rows:,}")
print(f"Train rows:              {len(df_train):,} ({len(df_train)/cleaned_rows:.2%})")
print(f"Validation rows:         {len(df_val):,} ({len(df_val)/cleaned_rows:.2%})")
print(f"Test rows:               {len(df_test):,} ({len(df_test)/cleaned_rows:.2%})")
print(f"Rows used in this run:   {len(df_train)+len(df_val)+len(df_test):,}")
print("Run mode:", RUN_MODE)
print("Full-data mode:", TRAIN_LIMIT is None and VAL_LIMIT is None and TEST_LIMIT is None)
print("NOTE: the raw CSV is not identical to the training set: only explicitly invalid/"
      "empty/caption-artifact rows are removed. INCLUDE is not merged because it is an "
      "isolated-sign dataset and would change the task definition.")


In [ ]:
# 3a. Record the video-level split in the knowledge repository.
train_videos = int(df_train["video_id"].nunique())
val_videos = int(df_val["video_id"].nunique())
test_videos = int(df_test["video_id"].nunique())

overlap_train_val = len(set(df_train["video_id"]) & set(df_val["video_id"]))
overlap_train_test = len(set(df_train["video_id"]) & set(df_test["video_id"]))
overlap_val_test = len(set(df_val["video_id"]) & set(df_test["video_id"]))
leakage_free = (overlap_train_val == overlap_train_test == overlap_val_test == 0)

knowledge_insert(
    "splits",
    ["run_id", "train_rows", "val_rows", "test_rows", "train_videos", "val_videos", "test_videos", "leakage_free"],
    [RUN_ID, len(df_train), len(df_val), len(df_test), train_videos, val_videos, test_videos, int(leakage_free)],
)
knowledge_observation("split_video_overlap", {
    "train_val": overlap_train_val,
    "train_test": overlap_train_test,
    "val_test": overlap_val_test,
})
print("Split metadata recorded. Leakage-free:", leakage_free)


## 4. Tokenizer

A simple whitespace tokenizer is retained for the core experiment so the notebook does not hide the translation problem behind a large pretrained language model.

The vocabulary is built from **training text only**. Unknown validation/test words map to `<unk>`.

In [ ]:
## 4. SentencePiece subword tokenizer

# The previous run exposed a fundamental problem with a one-token-per-word vocabulary:
# 59,742 output classes for 99,699 training examples. That makes the decoder's
# classification problem unnecessarily sparse and encourages generic-output collapse.
#
# Keep the complete usable iSign dataset, but represent English with a compact
# subword vocabulary. SentencePiece is trained ONLY on the training split, then
# frozen and exported as an inference artifact. This preserves the research
# experiment while making rare words compositional instead of automatic <unk>.
import sentencepiece as spm

TOKENIZER_DIR = REPORT_DIR / "tokenizer"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_PREFIX = TOKENIZER_DIR / "signsage_en_sp"
TOKENIZER_MODEL = TOKENIZER_PREFIX.with_suffix(".model")
TOKENIZER_VOCAB = TOKENIZER_PREFIX.with_suffix(".vocab")
TOKENIZER_CORPUS = TOKENIZER_DIR / "train_text.txt"

# Deterministic training corpus: one normalized target per line.
with open(TOKENIZER_CORPUS, "w", encoding="utf-8") as f:
    for text in df_train["text"].map(normalize_text):
        f.write(text.replace("\n", " ") + "\n")

if not TOKENIZER_MODEL.exists():
    spm.SentencePieceTrainer.train(
        input=str(TOKENIZER_CORPUS),
        model_prefix=str(TOKENIZER_PREFIX),
        vocab_size=TARGET_VOCAB_SIZE,
        model_type="unigram",
        character_coverage=1.0,
        normalization_rule_name="nmt_nfkc_cf",
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        pad_piece="<pad>", unk_piece="<unk>",
        bos_piece="<bos>", eos_piece="<eos>",
        hard_vocab_limit=False,
        shuffle_input_sentence=False,
        input_sentence_size=0,
        seed_sentencepiece_size=1000000,
        num_threads=max(1, min(8, os.cpu_count() or 1)),
    )

sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]

# Keep a simple token_to_id dictionary for the existing experiment database/export code.
vocab = {sp.id_to_piece(i): i for i in range(sp.get_piece_size())}
inv_vocab = {i: sp.id_to_piece(i) for i in range(sp.get_piece_size())}

def encode_text(text, max_len=MAX_TOKENS):
    ids = sp.encode(normalize_text(text), out_type=int, add_bos=True, add_eos=True)
    if len(ids) > max_len:
        ids = ids[:max_len-1] + [EOS_IDX]
    return ids

def decode_ids(ids):
    clean=[]
    for idx in ids:
        idx=int(idx)
        if idx == EOS_IDX:
            break
        if idx not in (BOS_IDX, PAD_IDX):
            clean.append(idx)
    if not clean:
        return ""
    return normalize_text(sp.decode(clean))

def _unk_rate(frame):
    total=0
    unk=0
    for text in frame.text:
        ids=sp.encode(normalize_text(text), out_type=int)
        total += len(ids)
        unk += sum(int(x)==UNK_IDX for x in ids)
    return unk/max(1,total)

val_unk_rate, test_unk_rate = _unk_rate(df_val), _unk_rate(df_test)
print("SentencePiece vocabulary size:", sp.get_piece_size())
print(f"Validation <unk> piece rate: {val_unk_rate:.4%}")
print(f"Test <unk> piece rate: {test_unk_rate:.4%}")
print(f"Tokenizer model: {TOKENIZER_MODEL}")
assert PAD_IDX == 0 and UNK_IDX == 1 and BOS_IDX == 2 and EOS_IDX == 3


In [ ]:
# 4a. SentencePiece target-length audit
_train_token_lengths = np.array([len(sp.encode(normalize_text(t), out_type=int)) + 2 for t in df_train["text"]], dtype=np.int32)
_val_token_lengths = np.array([len(sp.encode(normalize_text(t), out_type=int)) + 2 for t in df_val["text"]], dtype=np.int32)
_test_token_lengths = np.array([len(sp.encode(normalize_text(t), out_type=int)) + 2 for t in df_test["text"]], dtype=np.int32)
print("SentencePiece token length percentiles (train):",
      {p:int(np.percentile(_train_token_lengths,p)) for p in [50,90,95,99,100]})
print("Targets truncated by MAX_TOKENS:",
      int((_train_token_lengths > MAX_TOKENS).sum()),
      f"/ {len(_train_token_lengths):,} training rows")
assert (_train_token_lengths > MAX_TOKENS).sum() <= max(10, int(0.001 * len(_train_token_lengths))), \
    "Too many targets would be truncated; increase MAX_TOKENS before training."


## 5. Pose representation and face ablation

The actual pose header contains:

- `POSE_LANDMARKS`: 33
- `FACE_LANDMARKS`: 468
- `LEFT_HAND_LANDMARKS`: 21
- `RIGHT_HAND_LANDMARKS`: 21
- `POSE_WORLD_LANDMARKS`: 33

For SignSage's intended MediaPipe-style input, the main experiment uses the first **543 image-space landmarks** (pose + face + hands) and deliberately excludes the duplicated `POSE_WORLD_LANDMARKS` block.

The no-face ablation retains pose + both hands.

Sequences are normalized by a sequence-level body center and scale. This reduces dependence on camera translation/size while preserving temporal motion.

In [ ]:
# Confirm component ranges from the real file rather than hardcoding the face indices blindly.
def get_component_ranges(pose):
    ranges = {}
    idx = 0
    for component in pose.header.components:
        n_points = len(component.points)
        ranges[str(component.name)] = (idx, idx + n_points)
        idx += n_points
    return ranges

component_ranges = get_component_ranges(example_pose)
print(component_ranges)

IMAGE_KEYPOINTS = list(range(543))
face_range = component_ranges.get("FACE_LANDMARKS")
assert face_range is not None, "FACE_LANDMARKS was not found; do not run the ablation until the pose header is understood."

keep_noface = [
    i for i in IMAGE_KEYPOINTS
    if not (face_range[0] <= i < face_range[1])
]

FULL_FEATURE_MASK = np.array([3*i + j for i in IMAGE_KEYPOINTS for j in range(3)], dtype=np.int64)
NOFACE_FEATURE_MASK = np.array([3*i + j for i in keep_noface for j in range(3)], dtype=np.int64)

print("Full keypoints/features:", len(IMAGE_KEYPOINTS), len(FULL_FEATURE_MASK))
print("No-face keypoints/features:", len(keep_noface), len(NOFACE_FEATURE_MASK))


### 5a. Disk-backed normalized-pose cache

The cache stores normalized pose arrays locally after their first access. In **Kaggle mode**, this is a small working-disk cache over a remote source; it is intentionally populated lazily rather than by scanning the entire archive.


In [ ]:
# The normalized pose cache is regenerable scratch data, not a deliverable, so on
# Kaggle it deliberately does NOT live under DATA_DIR (/kaggle/working/...):
# /kaggle/working is committed as the notebook's saved output and counts against
# the 20 GB output limit, so a multi-GB cache there would bloat every saved version
# and compete with the checkpoints/reports that are actually meant to be downloaded.
# /kaggle/temp is session scratch on the same instance disk and is discarded when
# the session ends -- correct for this data, since every cached array can be
# rebuilt from the remote archive. Local mode keeps the cache next to the dataset,
# where persisting it across runs is the whole point.
CACHE_ROOT = Path("/kaggle/temp/signsage_cache") if DATA_MODE == "kaggle" else DATA_DIR / "cache"
POSE_CACHE_DIR = CACHE_ROOT / "normalized_v1"
POSE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Pose cache dir:", POSE_CACHE_DIR)

def pose_path_for_uid(uid):
    if DATA_MODE != "local":
        raise RuntimeError("pose_path_for_uid() is only valid in local mode; use read_pose_bytes_for_uid() in Kaggle mode.")
    path = POSE_DIR / f"{uid}.pose"
    if not path.is_file():
        raise FileNotFoundError(f"Missing UID-level pose file: {path}")
    return path

def _cache_path_for_uid(uid):
    return POSE_CACHE_DIR / f"{uid}.npy"

# Running counters so rare data-quality edge cases (missing hip landmarks, or a pose
# sequence with no frame having any spatial variance at all) are tallied across the
# whole run instead of only ever appearing as an unlabeled stderr warning. Reported
# in Section 17/19 below.
POSE_NORMALIZATION_STATS = {
    "total_sequences": 0,
    "bad_center_sequences": 0,
    "degenerate_scale_sequences": 0,
}

def normalize_pose_sequence(flat):
    """Root-center and scale one sequence; preserves frame-to-frame motion."""
    POSE_NORMALIZATION_STATS["total_sequences"] += 1
    x = flat.reshape(flat.shape[0], 543, 3).astype(np.float32)
    hip = x[:, [23, 24], :]
    center = hip.mean(axis=1, keepdims=True)
    bad_center = ~np.isfinite(center).all(axis=(1, 2))
    if bad_center.any():
        POSE_NORMALIZATION_STATS["bad_center_sequences"] += 1
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Mean of empty slice")
            center[bad_center] = np.nanmean(x[bad_center, :33, :], axis=1, keepdims=True)
    x = x - center
    body = x[:, :33, :]
    scale = np.nanstd(body.reshape(body.shape[0], -1), axis=1)
    valid_scale = scale[np.isfinite(scale) & (scale > 1e-6)]
    # BUG FIX: np.nanmedian on an empty array raises "Mean of empty slice" *unguarded* --
    # this used to leak into training-loop stderr (visible in the first Kaggle run).
    # A sequence hitting this path has NO frame with any real body-keypoint spread at all
    # (a fully static/degenerate capture), which is worth counting, not just silently
    # defaulting to scale=1.0 with no record of how often it happens.
    if valid_scale.size == 0:
        POSE_NORMALIZATION_STATS["degenerate_scale_sequences"] += 1
        scale = 1.0
    else:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Mean of empty slice")
            scale = np.nanmedian(valid_scale)
        if not np.isfinite(scale) or scale <= 1e-6:
            POSE_NORMALIZATION_STATS["degenerate_scale_sequences"] += 1
            scale = 1.0
    x = x / float(scale)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return x.reshape(x.shape[0], -1).astype(np.float32)

def resize_sequence(seq, target_len):
    src_len = seq.shape[0]
    if src_len <= 0:
        raise ValueError("Empty pose sequence")
    if src_len == target_len:
        return seq.astype(np.float32)
    if src_len > target_len:
        idx = np.linspace(0, src_len - 1, target_len)
        return seq[np.round(idx).astype(int)].astype(np.float32)
    out = np.zeros((target_len, seq.shape[1]), dtype=np.float32)
    out[:src_len] = seq
    return out

def _read_and_normalize_raw(uid):
    raw_bytes = read_pose_bytes_for_uid(uid)
    pose = Pose.read(raw_bytes)
    data = np.asarray(np.ma.filled(pose.body.data, 0.0), dtype=np.float32)
    if data.ndim != 4 or data.shape[1] < 1:
        raise ValueError(f"Unexpected pose tensor {data.shape} for {uid}")
    data = data[:, 0, :, :]
    data = data[:, :543, :]
    flat = data.reshape(data.shape[0], -1).astype(np.float32)
    return normalize_pose_sequence(flat)

@lru_cache(maxsize=CACHE_SIZE)
def _read_pose_file(uid):
    if USE_POSE_DISK_CACHE:
        cpath = _cache_path_for_uid(uid)
        if cpath.is_file():
            try:
                return np.load(cpath).astype(np.float32)
            except Exception:
                cpath.unlink(missing_ok=True)
    arr = _read_and_normalize_raw(uid)
    if USE_POSE_DISK_CACHE:
        tmp_path = _cache_path_for_uid(uid).with_name(_cache_path_for_uid(uid).name + ".tmp")
        with open(tmp_path, "wb") as f:
            np.save(f, arr.astype(np.float16))
        os.replace(tmp_path, _cache_path_for_uid(uid))
    return arr

def load_pose(row, feature_mask=FULL_FEATURE_MASK, max_frames=MAX_FRAMES):
    raw = _read_pose_file(row["uid"])
    raw = raw[:, feature_mask]
    true_len = min(len(raw), max_frames)
    processed = resize_sequence(raw, max_frames)
    return processed, true_len

for _, row in df_train.head(5).iterrows():
    if DATA_MODE == "local":
        p = pose_path_for_uid(row["uid"])
        print(row["uid"], "->", p.name)
    else:
        print(row["uid"], "-> remote ZIP member")
print("UID-to-pose alignment passed.")

if USE_POSE_DISK_CACHE:
    # Spot-check that downcasting the normalized cache to fp16 (to save disk)
    # doesn't meaningfully distort values relative to the fp32 array it was
    # computed from. `_read_pose_file` returns the fresh fp32 array on a
    # cache miss (not yet round-tripped), so the cache must be cleared and
    # re-read to actually exercise the on-disk fp16 path.
    print("\nSpot-checking fp16 disk-cache precision vs. fresh fp32 computation:")
    for _uid in df_train.head(3)["uid"].tolist():
        _fresh = _read_and_normalize_raw(_uid)
        _read_pose_file(_uid)          # populate the disk cache
        _read_pose_file.cache_clear()  # force the next call to read from disk, not memory
        _cached = _read_pose_file(_uid)
        _diff = float(np.max(np.abs(_fresh - _cached)))
        print(f"  {_uid}: max abs diff (fp32 fresh vs. fp16 disk round-trip) = {_diff:.6f}")
    print("Small diffs relative to the ~unit normalized-pose scale confirm fp16 caching is safe.\n")


**Before running a full pass**, a rough sanity check on free disk space for the new
cache (skip if `USE_POSE_DISK_CACHE = False`):


In [ ]:
import shutil

def _estimate_full_cache_bytes(n_rows, avg_frames=150, n_landmarks=543, bytes_per_val=2):
    return n_rows * avg_frames * n_landmarks * 3 * bytes_per_val

if USE_POSE_DISK_CACHE:
    # BUG FIX: this used to check disk_usage(DATA_DIR), but the pose cache actually lives
    # at POSE_CACHE_DIR -- /kaggle/temp/... on Kaggle, DATA_DIR/cache locally -- which is a
    # different mount from DATA_DIR on Kaggle. Checking DATA_DIR told you nothing about the
    # disk that actually fills up as this cache grows.
    free_bytes = shutil.disk_usage(POSE_CACHE_DIR).free
    projected_rows = len(df) if RUN_MODE == "full" else sum(x for x in [TRAIN_LIMIT, VAL_LIMIT, TEST_LIMIT] if x)
    projected_bytes = _estimate_full_cache_bytes(projected_rows)
    print(f"Free space at {POSE_CACHE_DIR}: {free_bytes / 1024**3:.1f} GB")
    print(f"Rough cache estimate for this run: {projected_bytes / 1024**3:.2f} GB")
    if projected_bytes > 0.8 * free_bytes:
        print("WARNING: estimated cache is too large for the available disk. Reduce sample limits.")
else:
    print("USE_POSE_DISK_CACHE is False -- skipping disk-space check.")


### 5c. Pose cache warm-up

**Disabled by default.** The previous version prewarmed every selected UID before training. That is reasonable only when the data is already local. In Kaggle mode, prewarming thousands of remote ZIP members just moves the same work earlier and makes the notebook appear stuck.

Training now fetches pose members lazily and shows progress. Set `PREWARM_CACHE = True` only if you deliberately want to populate the selected sample before training.


In [ ]:
if PREWARM_CACHE and USE_POSE_DISK_CACHE:
    _prewarm_uids = pd.concat([df_train["uid"], df_val["uid"], df_test["uid"]]).unique()
    _already_cached = sum(1 for u in _prewarm_uids if _cache_path_for_uid(u).is_file())
    print(f"{len(_prewarm_uids):,} unique selected rows; {_already_cached:,} already cached.")
    for uid in tqdm(_prewarm_uids, desc="Prewarming selected pose cache"):
        _read_pose_file(uid)
    print("Selected-cache prewarm complete.")
else:
    print("Lazy pose loading enabled — no full prewarm will run.")


In [ ]:
# 6. Sign → Text Dataset/DataLoader
class SignToTextDataset(Dataset):
    def __init__(self, frame, feature_mask=FULL_FEATURE_MASK):
        self.rows = frame.reset_index(drop=True)
        self.feature_mask = feature_mask
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        frames, length = load_pose(row, self.feature_mask)
        tokens = encode_text(row["text"])
        return (torch.from_numpy(np.ascontiguousarray(frames)).float(),
                torch.tensor(length, dtype=torch.long),
                torch.tensor(tokens, dtype=torch.long))

def collate_sign_text(batch):
    frames, lengths, tokens = zip(*batch)
    frames = torch.stack(frames)
    lengths = torch.stack(lengths)
    max_tok = max(map(len, tokens))
    padded = torch.full((len(tokens), max_tok), PAD_IDX, dtype=torch.long)
    for i,t in enumerate(tokens): padded[i,:len(t)] = t
    return frames, lengths, padded[:,:-1], padded[:,1:]

# The sampler groups samples by their physical ZIP position before shuffling
# batches. This reduces HTTP range churn while preserving stochastic batch order.
class ArchiveLocalBatchSampler(torch.utils.data.Sampler):
    """Shuffle batch order while keeping each batch physically local in the ZIP."""
    def __init__(self, dataset, batch_size, uid_to_offset, seed=42, drop_last=False):
        self.dataset = dataset
        self.batch_size = int(batch_size)
        self.seed = int(seed)
        self.drop_last = bool(drop_last)
        offsets = []
        for i, uid in enumerate(dataset.rows["uid"].astype(str)):
            off = uid_to_offset.get(uid)
            if off is None:
                raise KeyError(f"No archive offset for UID {uid}")
            offsets.append((int(off), i))
        offsets.sort(key=lambda x: x[0])
        ordered = [i for _, i in offsets]
        self.batches = [
            ordered[j:j+self.batch_size]
            for j in range(0, len(ordered), self.batch_size)
        ]
        if self.drop_last and self.batches and len(self.batches[-1]) < self.batch_size:
            self.batches.pop()
        self.epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        order = rng.permutation(len(self.batches))
        for j in order:
            batch = list(self.batches[int(j)])
            # Keep the batch's ZIP locality, but randomize member order inside it
            # so gradient batches are not identical to archive ordering.
            rng.shuffle(batch)
            yield batch

    def __len__(self): return len(self.batches)
    def set_epoch(self, epoch): self.epoch = int(epoch)

def _archive_offsets_for_frame(frame):
    if DATA_MODE != "kaggle" or not REMOTE_BATCH_LOCALITY:
        return {}
    offsets = {}
    for uid in frame["uid"].astype(str):
        info = remote_pose_store.info_for_uid(str(uid))
        offsets[str(uid)] = int(info.header_offset)
    return offsets

_loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
if NUM_WORKERS > 0:
    _loader_kwargs.update(persistent_workers=True, prefetch_factor=2)

train_ds_full = SignToTextDataset(df_train, FULL_FEATURE_MASK)
val_ds_full = SignToTextDataset(df_val, FULL_FEATURE_MASK)
test_ds_full = SignToTextDataset(df_test, FULL_FEATURE_MASK)

if DATA_MODE == "kaggle" and REMOTE_BATCH_LOCALITY:
    _train_offsets = _archive_offsets_for_frame(df_train)
    _val_offsets = _archive_offsets_for_frame(df_val)
    _test_offsets = _archive_offsets_for_frame(df_test)
    train_batch_sampler = ArchiveLocalBatchSampler(train_ds_full, BATCH_SIZE, _train_offsets, SEED)
    val_batch_sampler = ArchiveLocalBatchSampler(val_ds_full, BATCH_SIZE, _val_offsets, SEED)
    test_batch_sampler = ArchiveLocalBatchSampler(test_ds_full, BATCH_SIZE, _test_offsets, SEED)
    train_loader_full = DataLoader(train_ds_full, batch_sampler=train_batch_sampler,
                                   collate_fn=collate_sign_text, **_loader_kwargs)
    val_loader_full = DataLoader(val_ds_full, batch_sampler=val_batch_sampler,
                                 collate_fn=collate_sign_text, **_loader_kwargs)
    test_loader_full = DataLoader(test_ds_full, batch_sampler=test_batch_sampler,
                                  collate_fn=collate_sign_text, **_loader_kwargs)
    print("Remote archive locality batching: ENABLED")
else:
    train_loader_full = DataLoader(train_ds_full, batch_size=BATCH_SIZE, shuffle=True,
                                   collate_fn=collate_sign_text, **_loader_kwargs)
    val_loader_full = DataLoader(val_ds_full, batch_size=BATCH_SIZE, shuffle=False,
                                 collate_fn=collate_sign_text, **_loader_kwargs)
    test_loader_full = DataLoader(test_ds_full, batch_size=BATCH_SIZE, shuffle=False,
                                  collate_fn=collate_sign_text, **_loader_kwargs)

# Optional no-face ablation uses the same split and loader contract, with the verified
# no-face feature mask. It is OFF in the primary run.
train_ds_noface = SignToTextDataset(df_train, NOFACE_FEATURE_MASK)
val_ds_noface = SignToTextDataset(df_val, NOFACE_FEATURE_MASK)
test_ds_noface = SignToTextDataset(df_test, NOFACE_FEATURE_MASK)

train_loader_noface = DataLoader(
    train_ds_noface, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_sign_text, **_loader_kwargs
)
val_loader_noface = DataLoader(
    val_ds_noface, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_sign_text, **_loader_kwargs
)
test_loader_noface = DataLoader(
    test_ds_noface, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_sign_text, **_loader_kwargs
)

LANDMARK_DIM = len(FULL_FEATURE_MASK)
NOFACE_DIM = len(NOFACE_FEATURE_MASK)
print("LANDMARK_DIM:", LANDMARK_DIM, "| NOFACE_DIM:", NOFACE_DIM)
print("Training examples this run:", len(train_ds_full))
print("Training batches/epoch:", len(train_loader_full))


In [ ]:
knowledge_observation("configuration", {
    "seed": SEED, "run_mode": RUN_MODE, "data_mode": DATA_MODE,
    "primary_sign_to_text": RUN_PRIMARY_S2T,
    "full_usable_dataset_requested": TRAIN_LIMIT is None,
    "train_limit": TRAIN_LIMIT, "val_limit": VAL_LIMIT, "test_limit": TEST_LIMIT,
    "epochs_transformer": N_EPOCHS_TRANSFORMER,
    "max_tokens": MAX_TOKENS, "max_frames": MAX_FRAMES,
    "batch_size": BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
    "amp": bool(USE_AMP), "full_feature_dim": int(LANDMARK_DIM),
    "no_face_feature_dim": int(NOFACE_DIM),
    "vocab_size": int(len(vocab)),
    "run_gru_baseline": RUN_GRU_BASELINE,
    "run_noface_ablation": RUN_NOFACE_ABLATION,
    "run_text_to_pose": RUN_TEXT_TO_POSE,
})
with open(REPORT_DIR / "token_vocabulary.json","w",encoding="utf-8") as f:
    json.dump({"token_to_id":vocab,"special_tokens":SPECIALS},f,indent=2,ensure_ascii=False)
knowledge_artifact("token_vocabulary","json",REPORT_DIR / "token_vocabulary.json",
                   "Sign-to-text token mapping used by the primary experiment.")
print("Primary experiment configuration recorded.")


## 6a. One-batch smoke check

This catches the exact class of failure that previously appeared as a delayed `FileNotFoundError` or undefined dimension.

In [ ]:
batch = next(iter(train_loader_full))
frames_b, lengths_b, tok_in_b, tok_out_b = batch
print("frames:", tuple(frames_b.shape))
print("lengths:", lengths_b.tolist())
print("tok_in:", tuple(tok_in_b.shape))
print("tok_out:", tuple(tok_out_b.shape))
assert frames_b.shape[-1] == LANDMARK_DIM
assert torch.isfinite(frames_b).all()
print("DataLoader smoke check passed.")


In [ ]:
# 6b. Remote data-loader performance preflight
# Measure the actual pose-delivery path, but do not abort the full run solely because
# the remote source exceeds the old 5 s/batch target.
assert len(train_loader_full.dataset) == len(df_train)
assert len(val_loader_full.dataset) == len(df_val)
assert len(test_loader_full.dataset) == len(df_test)

if DATA_MODE == "kaggle" and RUN_PRIMARY_S2T:
    import time as _perf_time
    _t0 = _perf_time.perf_counter()
    _n = 0
    _sample_count = 0
    for _batch in train_loader_full:
        _n += 1
        _sample_count += int(_batch[0].shape[0])
        if _n >= PERF_PREFLIGHT_BATCHES:
            break
    _elapsed = _perf_time.perf_counter() - _t0
    _sec_per_batch = _elapsed / max(1, _n)
    _samples_per_sec = _sample_count / max(1e-9, _elapsed)
    _batches_per_epoch = len(train_loader_full)
    _loader_hours = _batches_per_epoch * _sec_per_batch / 3600.0
    print(f"Data-loader preflight: {_n} batches, {_elapsed:.1f}s total, "
          f"{_sec_per_batch:.2f}s/batch, {_samples_per_sec:.2f} samples/s")
    print(f"Approximate train-loader floor: {_loader_hours:.2f} h/epoch "
          f"({len(train_loader_full):,} batches; model compute adds overhead).")
    if _sec_per_batch > PERF_MAX_SECONDS_PER_BATCH:
        print(
            f"WARNING: remote path is slower than the old {PERF_MAX_SECONDS_PER_BATCH:.2f}s/batch "
            "target, but the FULL-CORPUS run will continue. This check is informational."
        )
else:
    print("Remote performance preflight skipped outside Kaggle mode.")


## 7. Sign → Text: GRU baseline

This is intentionally simple. It is a sanity baseline, not the final model.

The important criterion is not whether it reaches a strong score; it is whether the training signal is learnable without immediately overfitting into nonsense.

In [ ]:
## 7. Optional GRU baseline
# Disabled by default for the full run so Kaggle compute is spent on the actual research target.
if RUN_GRU_BASELINE:
    class GRUEncoder(nn.Module):
        def __init__(self,input_dim,hidden_dim=192):
            super().__init__(); self.gru=nn.GRU(input_dim,hidden_dim,batch_first=True,bidirectional=True); self.proj=nn.Linear(hidden_dim*2,hidden_dim)
        def forward(self,x,lengths):
            packed=nn.utils.rnn.pack_padded_sequence(x,lengths.cpu().clamp(min=1),batch_first=True,enforce_sorted=False)
            _,h=self.gru(packed); h=torch.cat([h[0],h[1]],dim=-1); return self.proj(h).unsqueeze(0)
    class GRUDecoder(nn.Module):
        def __init__(self,vocab_size,hidden_dim=192):
            super().__init__(); self.embed=nn.Embedding(vocab_size,hidden_dim,padding_idx=PAD_IDX); self.gru=nn.GRU(hidden_dim,hidden_dim,batch_first=True); self.out=nn.Linear(hidden_dim,vocab_size)
        def forward(self,tokens,hidden):
            out,hidden=self.gru(self.embed(tokens),hidden); return self.out(out),hidden
    class Seq2SeqGRU(nn.Module):
        def __init__(self,input_dim,vocab_size,hidden_dim=192):
            super().__init__(); self.encoder=GRUEncoder(input_dim,hidden_dim); self.decoder=GRUDecoder(vocab_size,hidden_dim)
        def forward(self,frames,lengths,target_in):
            return self.decoder(target_in,self.encoder(frames,lengths))[0]
    @torch.no_grad()
    def eval_loss_gru(model,loader,criterion):
        model.eval(); total=0.0
        for frames,lengths,tin,tout in tqdm(loader,desc="GRU eval",leave=False):
            frames,lengths,tin,tout=(frames.to(DEVICE,non_blocking=True),
                                     lengths.to(DEVICE,non_blocking=True),
                                     tin.to(DEVICE,non_blocking=True),
                                     tout.to(DEVICE,non_blocking=True))
            with autocast_ctx():
                logits=model(frames,lengths,tin)
                total += criterion(logits.reshape(-1,len(vocab)),tout.reshape(-1)).item()
        return total/max(1,len(loader))

    def train_epoch_gru(model,loader,opt,criterion):
        model.train(); total=0
        for frames,lengths,tin,tout in tqdm(loader,desc="GRU train",leave=False):
            frames,lengths,tin,tout=frames.to(DEVICE),lengths.to(DEVICE),tin.to(DEVICE),tout.to(DEVICE)
            with autocast_ctx(): loss=criterion(model(frames,lengths,tin).reshape(-1,len(vocab)),tout.reshape(-1))
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); total+=loss.item()
        return total/max(1,len(loader))
    @torch.no_grad()
    def batch_greedy_gru(model,loader):
        model.eval(); refs=[]; hyps=[]; confs=[]
        for frames,lengths,tin,tout in tqdm(loader,desc="GRU test",leave=False):
            frames,lengths=frames.to(DEVICE,non_blocking=True),lengths.to(DEVICE,non_blocking=True)
            memory=model.encoder(frames,lengths)
            ids=torch.full((frames.size(0),1),BOS_IDX,dtype=torch.long,device=DEVICE)
            finished=torch.zeros(frames.size(0),dtype=torch.bool,device=DEVICE)
            sum_logp=torch.zeros(frames.size(0),device=DEVICE); steps=torch.zeros(frames.size(0),device=DEVICE)
            generated=[]
            hidden=memory
            for step in range(MAX_DECODE_TOKENS):
                logits,hidden=model.decoder(ids[:,-1:],hidden)
                logits=logits[:,-1].float()
                logits[:,PAD_IDX]=-torch.inf; logits[:,BOS_IDX]=-torch.inf; logits[:,UNK_IDX]=-torch.inf
                if step < MIN_OUTPUT_TOKENS: logits[:,EOS_IDX]=-torch.inf
                lp=torch.log_softmax(logits,dim=-1); prob,nxt=torch.max(lp,dim=-1)
                active=~finished; sum_logp += torch.where(active,prob,torch.zeros_like(prob)); steps += active
                nxt=torch.where(finished,torch.full_like(nxt,EOS_IDX),nxt)
                generated.append(nxt.cpu().numpy()); finished |= nxt.eq(EOS_IDX)
                ids=torch.cat([ids,nxt[:,None]],dim=1)
                if finished.all(): break
            gen=np.stack(generated,axis=1)
            conf=torch.exp(sum_logp/steps.clamp(min=1)).cpu().numpy()
            for r,row in enumerate(tout.cpu().tolist()):
                refs.append(decode_ids(row)); hyps.append(decode_ids(gen[r].tolist())); confs.append(float(conf[r]))
        return refs,hyps,confs

    gru_model=Seq2SeqGRU(LANDMARK_DIM,len(vocab)).to(DEVICE)
    gru_opt=torch.optim.AdamW(gru_model.parameters(),lr=1e-3,weight_decay=1e-4)
    criterion_gru=nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    gru_history=[]; best=float("inf")
    for epoch in range(1,max(1,N_EPOCHS_GRU)+1):
        tr=train_epoch_gru(gru_model,train_loader_full,gru_opt,criterion_gru)
        va=eval_loss_gru(gru_model,val_loader_full,criterion_gru)
        gru_history.append((epoch,tr,va)); print(f"[GRU] {epoch}: train={tr:.4f} val={va:.4f}")
        if va<best: best=va; torch.save(gru_model.state_dict(),CHECKPOINT_DIR/"signsage_gru_best.pt")
    gru_model.load_state_dict(torch.load(CHECKPOINT_DIR/"signsage_gru_best.pt",map_location=DEVICE))
else:
    gru_model=None; gru_history=[]
    print("GRU baseline skipped; primary compute is reserved for Sign→Text Transformer.")


In [ ]:
# 7a. Record optional GRU experiment.
if RUN_GRU_BASELINE:
    _GRU_NAME="GRU baseline"; gru_ckpt=CHECKPOINT_DIR/"signsage_gru_best.pt"
    gru_best_val=float(min(v for _,_,v in gru_history))
    knowledge_insert("experiments",
        ["run_id","name","family","input_features","epochs","batch_size","learning_rate","weight_decay","checkpoint_path","best_validation_loss"],
        [RUN_ID,_GRU_NAME,"Sign→Text","full_pose",N_EPOCHS_GRU,BATCH_SIZE,1e-3,1e-4,str(gru_ckpt),gru_best_val],
        delete_keys={"run_id":RUN_ID,"name":_GRU_NAME})
    for epoch,tr_loss,val_loss in gru_history:
        knowledge_metric(_GRU_NAME,f"train_loss_epoch_{epoch}",tr_loss,"train")
        knowledge_metric(_GRU_NAME,f"val_loss_epoch_{epoch}",val_loss,"validation")
    knowledge_artifact(_GRU_NAME,"checkpoint",gru_ckpt,"Optional GRU Sign→Text checkpoint.")
else:
    print("GRU metadata skipped.")


## 8. Sign → Text: spatial-temporal Transformer

The encoder sees the full landmark vector at every frame; temporal self-attention then relates frames across the sequence. This is the plain Transformer requested by the project decision log; GAT is intentionally not introduced here.

In [ ]:
## 8. Primary model: spatial-temporal Transformer

class FrameProjection(nn.Module):
    """Project spatial landmarks plus explicit frame-to-frame motion features."""
    def __init__(self,input_dim,d_model):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(384,d_model),
            nn.LayerNorm(d_model),
        )
        self.temporal=nn.Conv1d(d_model,d_model,kernel_size=5,padding=2,groups=d_model)
    def forward(self,x):
        # Explicit velocity is zero for the first frame. This makes the temporal
        # signal available before attention rather than asking the encoder to infer
        # all derivatives from absolute coordinates alone.
        delta = torch.zeros_like(x)
        delta[:,1:] = x[:,1:] - x[:,:-1]
        y=self.net(torch.cat([x, delta], dim=-1))
        y=self.temporal(y.transpose(1,2)).transpose(1,2)
        return y

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len):
        super().__init__()
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class SignTransformer(nn.Module):
    def __init__(self,input_dim,vocab_size,d_model=192,nhead=4,layers=3,ff=512,dropout=0.15):
        super().__init__()
        self.input_proj=FrameProjection(input_dim,d_model)
        self.src_pos=PositionalEncoding(d_model,MAX_FRAMES+4)
        self.tok_embed=nn.Embedding(vocab_size,d_model,padding_idx=PAD_IDX)
        self.tgt_pos=PositionalEncoding(d_model,MAX_TOKENS+4)
        enc=nn.TransformerEncoderLayer(d_model,nhead,ff,dropout,batch_first=True,norm_first=False)
        dec=nn.TransformerDecoderLayer(d_model,nhead,ff,dropout,batch_first=True,norm_first=False)
        self.encoder=nn.TransformerEncoder(enc,layers)
        self.decoder=nn.TransformerDecoder(dec,layers)
        self.out=nn.Linear(d_model,vocab_size)
        self.scale=d_model**0.5
    def encode(self,frames,src_pad):
        return self.encoder(self.src_pos(self.input_proj(frames)),src_key_padding_mask=src_pad)
    def decode(self,tokens,memory,tgt_mask,src_pad,tgt_pad=None):
        x=self.tgt_pos(self.tok_embed(tokens)*self.scale)
        return self.out(self.decoder(
            x,memory,tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        ))
    def forward(self,frames,src_pad,tokens,tgt_mask):
        return self.decode(
            tokens,self.encode(frames,src_pad),tgt_mask,src_pad,
            tgt_pad=tokens.eq(PAD_IDX)
        )

def padding_mask(lengths,max_len):
    return torch.arange(max_len,device=lengths.device).unsqueeze(0)>=lengths.unsqueeze(1)

def causal_mask(n):
    return torch.triu(torch.ones(n,n,device=DEVICE,dtype=torch.bool),diagonal=1)

def train_epoch_transformer(model,loader,opt,criterion,epoch=0):
    model.train(); total=0.0; opt.zero_grad(set_to_none=True)
    if hasattr(getattr(loader, 'batch_sampler', None), 'set_epoch'):
        loader.batch_sampler.set_epoch(epoch)
    for i,(frames,lengths,tin,tout) in enumerate(tqdm(loader,desc="Transformer train",leave=False)):
        frames,lengths,tin,tout=(frames.to(DEVICE, non_blocking=True), lengths.to(DEVICE, non_blocking=True), tin.to(DEVICE, non_blocking=True), tout.to(DEVICE, non_blocking=True))
        src_pad=padding_mask(lengths,frames.size(1)); tgt_mask=causal_mask(tin.size(1))
        with autocast_ctx():
            logits=model(frames,src_pad,tin,tgt_mask)
            loss=criterion(logits.reshape(-1,logits.size(-1)),tout.reshape(-1))
        if not torch.isfinite(loss): raise RuntimeError("Non-finite Transformer loss.")
        scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
        if (i+1)%GRAD_ACCUM_STEPS==0 or i==len(loader)-1:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        total+=loss.item()
        if (i + 1) % 250 == 0:
            # Network-backed datasets can fail transiently. Keep the most recent
            # optimizer/model state so a completed portion of an epoch is not lost.
            torch.save({
                "state_dict": model.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "epoch": epoch,
                "batch": i + 1,
                "model_config": {
                    "input_dim": model.input_proj.net[0].in_features,
                    "raw_feature_dim": model.input_proj.net[0].in_features//2,
                    "vocab_size": len(vocab),
                    "d_model": 192, "nhead": 4, "layers": 3, "ff": 512,
                    "dropout": 0.15, "max_frames": MAX_FRAMES, "max_tokens": MAX_TOKENS,
                },
                "vocab": vocab,
            }, CHECKPOINT_DIR / "signsage_transformer_recovery.pt")
    return total/max(1,len(loader))

@torch.no_grad()
def eval_loss_transformer(model,loader,criterion):
    model.eval(); total=0.0
    for frames,lengths,tin,tout in tqdm(loader,desc="Transformer eval",leave=False):
        frames,lengths,tin,tout=(frames.to(DEVICE, non_blocking=True), lengths.to(DEVICE, non_blocking=True), tin.to(DEVICE, non_blocking=True), tout.to(DEVICE, non_blocking=True))
        with autocast_ctx():
            logits=model(frames,padding_mask(lengths,frames.size(1)),tin,causal_mask(tin.size(1)))
            total+=criterion(logits.reshape(-1,logits.size(-1)),tout.reshape(-1)).item()
    return total/max(1,len(loader))

def _ban_repeated_ngram(logits,ids,n):
    if n<=1 or len(ids)<n-1: return logits
    prefix=tuple(ids[-(n-1):])
    for j in range(len(ids)-n+1):
        ng=tuple(ids[j:j+n])
        if ng[:-1]==prefix: logits[ng[-1]]=-torch.inf
    return logits

@torch.no_grad()
def constrained_greedy_transformer(model,frames,length,return_confidence=False):
    model.eval(); frames=frames.unsqueeze(0).to(DEVICE)
    length=torch.tensor([int(length)],device=DEVICE)
    src_pad=padding_mask(length,frames.size(1)); memory=model.encode(frames,src_pad)
    ids=[BOS_IDX]; logps=[]
    for step in range(MAX_DECODE_TOKENS):
        tok=torch.tensor([ids],device=DEVICE)
        logits=model.decode(tok,memory,causal_mask(tok.size(1)),src_pad)[0,-1].float()
        logits[PAD_IDX]=-torch.inf; logits[BOS_IDX]=-torch.inf; logits[UNK_IDX]=-torch.inf
        if step < MIN_OUTPUT_TOKENS: logits[EOS_IDX]=-torch.inf
        logits=_ban_repeated_ngram(logits,ids,NO_REPEAT_NGRAM_SIZE)
        # Mild repetition penalty; do not use an aggressive penalty that can distort fluent English.
        for tid in set(ids[1:]):
            logits[tid]=logits[tid]/REPETITION_PENALTY if logits[tid]>=0 else logits[tid]*REPETITION_PENALTY
        lp=torch.log_softmax(logits,dim=-1)
        prob,tid=torch.max(lp,dim=-1)
        tid=int(tid); logps.append(float(prob))
        ids.append(tid)
        if tid==EOS_IDX: break
    text=decode_ids(ids[1:])
    conf=float(np.exp(np.mean(logps))) if logps else 0.0
    return (text,conf) if return_confidence else text

@torch.no_grad()
def batch_greedy_transformer(model,loader):
    """Fast full-test decoder with batch-safe constrained greedy decoding."""
    model.eval(); refs=[]; hyps=[]; confs=[]
    for frames,lengths,tin,tout in tqdm(loader,desc="Transformer test",leave=False):
        frames,lengths=frames.to(DEVICE,non_blocking=True),lengths.to(DEVICE,non_blocking=True)
        src_pad=padding_mask(lengths,frames.size(1))
        memory=model.encode(frames,src_pad)
        ids=torch.full((frames.size(0),1),BOS_IDX,dtype=torch.long,device=DEVICE)
        finished=torch.zeros(frames.size(0),dtype=torch.bool,device=DEVICE)
        sum_logp=torch.zeros(frames.size(0),dtype=torch.float32,device=DEVICE)
        active_steps=torch.zeros(frames.size(0),dtype=torch.long,device=DEVICE)
        generated=[]
        for step in range(MAX_DECODE_TOKENS):
            logits=model.decode(ids,memory,causal_mask(ids.size(1)),src_pad)[:,-1].float()
            logits[:,PAD_IDX]=-torch.inf; logits[:,BOS_IDX]=-torch.inf; logits[:,UNK_IDX]=-torch.inf
            if step < MIN_OUTPUT_TOKENS: logits[:,EOS_IDX]=-torch.inf
            lp=torch.log_softmax(logits,dim=-1)
            prob,nxt=torch.max(lp,dim=-1)
            # Only accumulate confidence for sequences that were still active.
            active=~finished
            sum_logp += torch.where(active,prob,torch.zeros_like(prob))
            active_steps += active.long()
            nxt=torch.where(finished,torch.full_like(nxt,EOS_IDX),nxt)
            generated.append(nxt.detach().cpu().numpy())
            finished |= nxt.eq(EOS_IDX)
            ids=torch.cat([ids,nxt[:,None]],dim=1)
            if finished.all(): break
        gen=np.stack(generated,axis=1)
        conf=torch.exp(sum_logp/active_steps.clamp(min=1)).detach().cpu().numpy()
        for r,row in enumerate(tout.cpu().tolist()):
            refs.append(decode_ids(row))
            hyps.append(decode_ids(gen[r].tolist()))
            confs.append(float(conf[r]))
    return refs,hyps,confs

def beam_transformer(model,frames,length,beam_width=BEAM_WIDTH):
    # Used only for qualitative inspection, not full-test metrics.
    model.eval(); frames=frames.unsqueeze(0).to(DEVICE); length=torch.tensor([int(length)],device=DEVICE)
    memory=model.encode(frames,padding_mask(length,frames.size(1)))
    beams=[(0.0,[BOS_IDX])]
    for step in range(MAX_DECODE_TOKENS):
        cand=[]
        for score,ids in beams:
            tok=torch.tensor([ids],device=DEVICE)
            logits=model.decode(tok,memory,causal_mask(tok.size(1)),padding_mask(length,frames.size(1)))[0,-1].float()
            logits[PAD_IDX]=-torch.inf; logits[BOS_IDX]=-torch.inf; logits[UNK_IDX]=-torch.inf
            if step<MIN_OUTPUT_TOKENS: logits[EOS_IDX]=-torch.inf
            logits=_ban_repeated_ngram(logits,ids,NO_REPEAT_NGRAM_SIZE)
            lp=torch.log_softmax(logits,dim=-1)
            vals,tids=torch.topk(lp,beam_width)
            for v,t in zip(vals.tolist(),tids.tolist()):
                new=ids+[int(t)]
                cand.append((score+float(v),new))
        cand.sort(key=lambda z:z[0]/max(1,len(z[1])-1),reverse=True)
        beams=cand[:beam_width]
        if all(x[1][-1]==EOS_IDX for x in beams): break
    best=max(beams,key=lambda z:z[0]/max(1,len(z[1])-1))
    return decode_ids(best[1][1:])

def fit_transformer(train_loader,val_loader,input_dim,checkpoint):
    model=SignTransformer(input_dim * 2,len(vocab)).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    # No label smoothing: preserve token-level learning signal while diagnosing collapse.
    crit=nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    best=float("inf"); bad=0; history=[]

    # A fresh run is intentional. The notebook never silently resumes a stale
    # checkpoint from an earlier dataset/configuration.
    for epoch in range(1,N_EPOCHS_TRANSFORMER+1):
        tr=train_epoch_transformer(model,train_loader,opt,crit,epoch)
        va=eval_loss_transformer(model,val_loader,crit)
        history.append((epoch,tr,va)); print(f"[Transformer] epoch {epoch:02d}: train={tr:.4f} val={va:.4f}")
        latest_payload={
            "state_dict":model.state_dict(),
            "epoch":epoch,
            "val_loss":va,
            "model_config":{
                "input_dim":input_dim*2,"raw_feature_dim":input_dim,"vocab_size":len(vocab),
                "d_model":192,"nhead":4,"layers":3,"ff":512,"dropout":0.15,
                "max_frames":MAX_FRAMES,"max_tokens":MAX_TOKENS,
            },
            "vocab":vocab,
        }
        torch.save(latest_payload,checkpoint.with_suffix(".latest.pt"))
        if va < best-1e-4:
            best=va; bad=0
            torch.save(latest_payload,checkpoint)
        else:
            bad+=1
            # With only two default full-data epochs, do not stop before the
            # requested complete passes have actually been completed.
            if bad>=3 and epoch < N_EPOCHS_TRANSFORMER:
                print("Early stopping.")
                break
        maybe_check_memory()
    best_payload=torch.load(checkpoint,map_location=DEVICE)
    model.load_state_dict(best_payload["state_dict"])
    return model,history

if RUN_PRIMARY_S2T:
    transformer_full,transformer_full_history=fit_transformer(
        train_loader_full,val_loader_full,LANDMARK_DIM,CHECKPOINT_DIR/"signsage_transformer_full.pt")
else:
    transformer_full=None; transformer_full_history=[]


## 9. Non-manual-marker ablation

This is meaningful only because the real pose header exposed `FACE_LANDMARKS` and the no-face mask is therefore verified rather than guessed.

In [ ]:
## 9. Optional non-manual-marker ablation
if RUN_NOFACE_ABLATION:
    transformer_noface, transformer_noface_history = fit_transformer(
        train_loader_noface, val_loader_noface, NOFACE_DIM,
        CHECKPOINT_DIR/"signsage_transformer_noface.pt")
else:
    transformer_noface=None; transformer_noface_history=[]
    print("No-face ablation skipped in the primary run; enable RUN_NOFACE_ABLATION for the thesis ablation experiment.")


In [ ]:
# 9a. Record Transformer experiment metadata.
def record_training_history(name,family,input_features,history,checkpoint_path,epochs):
    best_val=float(min(v for _,_,v in history)) if history else float("nan")
    knowledge_insert("experiments",
        ["run_id","name","family","input_features","epochs","batch_size","learning_rate","weight_decay","checkpoint_path","best_validation_loss"],
        [RUN_ID,name,family,input_features,epochs,BATCH_SIZE,2e-4,1e-4,str(checkpoint_path),best_val],
        delete_keys={"run_id":RUN_ID,"name":name})
    for epoch,tr_loss,val_loss in history:
        knowledge_metric(name,f"train_loss_epoch_{epoch}",tr_loss,"train")
        knowledge_metric(name,f"val_loss_epoch_{epoch}",val_loss,"validation")
    knowledge_artifact(name,"checkpoint",checkpoint_path,f"{name} checkpoint.")

if RUN_PRIMARY_S2T:
    record_training_history("Transformer Sign→Text","Sign→Text",LANDMARK_DIM,
                            transformer_full_history,CHECKPOINT_DIR/"signsage_transformer_full.pt",
                            N_EPOCHS_TRANSFORMER)
if RUN_NOFACE_ABLATION:
    record_training_history("Transformer (no face)","Sign→Text",NOFACE_DIM,
                            transformer_noface_history,CHECKPOINT_DIR/"signsage_transformer_noface.pt",
                            N_EPOCHS_TRANSFORMER)
print("Training metadata recorded.")


## 10. Held-out Sign → Text evaluation

Model selection uses validation loss. The numbers below are computed on the **test split** that was never used for checkpoint selection.

The trivial baseline predicts the most common training sentence. It is deliberately weak but necessary as a reference point.

In [ ]:
## 10. Held-out Sign → Text evaluation

# Full test split is evaluated. Fast batched greedy decoding is used for metrics;
# beam search is used only for qualitative inspection so evaluation does not become
# an unnecessary multi-hour bottleneck.
import jiwer, sacrebleu
from rouge_score import rouge_scorer
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def metrics(refs,hyps):
    return {
        "WER":float(jiwer.wer(refs,hyps)),
        "BLEU":float(sacrebleu.corpus_bleu(hyps,[refs]).score),
        "ROUGE-L":float(np.mean([rouge.score(r,h)["rougeL"].fmeasure for r,h in zip(refs,hyps)])),
        "exact_match":float(np.mean([normalize_text(r)==normalize_text(h) for r,h in zip(refs,hyps)])),
        "nonempty_output":float(np.mean([bool(normalize_text(h)) for h in hyps])),
    }

if transformer_full is None:
    raise RuntimeError("Primary Transformer was not trained.")

refs,tf_h,tf_conf=batch_greedy_transformer(transformer_full,test_loader_full)
results_s2t={"Transformer Sign→Text":metrics(refs,tf_h)}

if RUN_GRU_BASELINE:
    refs_gru,gru_h,gru_conf=batch_greedy_gru(gru_model,test_loader_full)
    results_s2t["GRU baseline"]=metrics(refs_gru,gru_h)
if RUN_NOFACE_ABLATION:
    refs_nf,nf_h,nf_conf=batch_greedy_transformer(transformer_noface,test_loader_noface)
    results_s2t["Transformer (no face)"]=metrics(refs_nf,nf_h)

baseline_pool=df_train.loc[df_train["n_words"]>=2, "text"].map(normalize_text)
if baseline_pool.empty:
    raise RuntimeError("No multi-word training targets remain for the trivial baseline.")
common_sentence=baseline_pool.value_counts().idxmax()
trivial_h=[common_sentence]*len(refs)
results_s2t["Trivial baseline"]=metrics(refs,trivial_h)

print(f"Evaluated {len(refs):,} held-out test examples.")
print(f"{'Model':<28}{'WER':>10}{'BLEU':>10}{'ROUGE-L':>12}{'Exact':>10}")
for name,m in results_s2t.items():
    print(f"{name:<28}{m['WER']:>10.3f}{m['BLEU']:>10.2f}{m['ROUGE-L']:>12.3f}{m['exact_match']:>10.2%}")

tf_unique=len(set(tf_h)); tf_unique_frac=tf_unique/max(1,len(tf_h))
print(f"\nTransformer output diversity: {tf_unique:,}/{len(tf_h):,} ({tf_unique_frac:.2%})")
print(f"Mean sequence confidence proxy: {np.mean(tf_conf):.4f}")
print(f"10th percentile confidence: {np.percentile(tf_conf,10):.4f}")
from collections import Counter as _Counter
_top_outputs=_Counter(tf_h).most_common(10)
print("Top generated outputs:", _top_outputs)
print("IMPORTANT: confidence is a diagnostic, not a calibrated probability.")
knowledge_observation("sign_to_text_evaluation",{
    "n_test":len(refs),"metrics":results_s2t,
    "unique_outputs":tf_unique,"unique_fraction":tf_unique_frac,
    "mean_confidence_proxy":float(np.mean(tf_conf)),
    "p10_confidence_proxy":float(np.percentile(tf_conf,10)),
})


# Release large CPU-side evaluation objects before subsequent notebook phases.
# Keeping full test refs/hypotheses plus live DataLoader workers alive can accumulate
# host RAM on Kaggle. The scalar metrics and diagnostic summary above are preserved.
try:
    del refs, tf_h, tf_conf, trivial_h
except NameError:
    pass
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("Post-test memory cleanup completed.")


In [ ]:
# 10a. Record Sign→Text evaluation metrics.
for model_name,metric_dict in results_s2t.items():
    for metric_name,metric_value in metric_dict.items():
        knowledge_metric(model_name,metric_name,metric_value,"test")
print("Sign→Text evaluation metrics recorded.")


### 10b. Validation-only confidence calibration

The runtime abstention threshold is calibrated on validation data, never on the held-out test set. It is a safety gate, not a claim that confidence is a true probability.

In [ ]:
def token_overlap_score(ref,hyp):
    r=normalize_text(ref).split(); h=normalize_text(hyp).split()
    if not r or not h:return 0.0
    from collections import Counter
    common=sum((Counter(r)&Counter(h)).values())
    return common/max(1,len(r))

cal_n=min(500,len(df_val))
cal_df=df_val.sample(cal_n,random_state=SEED).reset_index(drop=True)
cal_ds=SignToTextDataset(cal_df,FULL_FEATURE_MASK)
cal_loader=DataLoader(cal_ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_sign_text,**_loader_kwargs)
cal_refs,cal_hyps,cal_conf=batch_greedy_transformer(transformer_full,cal_loader)
cal_overlap=np.array([token_overlap_score(r,h) for r,h in zip(cal_refs,cal_hyps)])
good_conf=np.array(cal_conf)[cal_overlap>=0.5]
if len(good_conf)>=20:
    LIVE_CONFIDENCE_FLOOR=float(np.percentile(good_conf,10))
else:
    LIVE_CONFIDENCE_FLOOR=float(np.percentile(cal_conf,50)) if cal_conf else 0.0
    print("WARNING: too few validation examples met the overlap criterion; confidence gate is only a weak diagnostic.")
print(f"Validation calibration examples: {cal_n}")
print(f"LIVE_CONFIDENCE_FLOOR: {LIVE_CONFIDENCE_FLOOR:.4f}")
knowledge_observation("live_confidence_calibration",{
    "n_validation":cal_n,"good_overlap_examples":int(len(good_conf)),
    "threshold":LIVE_CONFIDENCE_FLOOR,
    "method":"10th percentile confidence among validation predictions with token overlap >= 0.5",
})


try:
    del cal_refs, cal_hyps, cal_conf, cal_overlap, cal_df, cal_ds, cal_loader
except NameError:
    pass
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Qualitative inspection: use beam search on a small, deterministic test sample.
for i in range(min(5,len(test_ds_full))):
    frames,length,tokens=test_ds_full[i]
    ref=decode_ids(tokens.tolist())
    hyp=beam_transformer(transformer_full,frames,length)
    print(f"[{i:02d}] REF : {ref}")
    print(f"     PRED: {hyp}")
    print()


## 11. Text → Sign: gloss and retrieval baseline

The naive gloss function is intentionally deterministic. The optional Groq function is not required for the core experiment and is never allowed to break the notebook.

In [ ]:
STOPWORDS = {
    "a","an","the","is","are","was","were","am","to","of","in","on","and","but",
    "it","this","that","for","with","as","at","by"
}

def text_to_naive_gloss(text):
    words = str(text).lower().split()
    kept = [w for w in words if w not in STOPWORDS]
    return " ".join(kept) if kept else " ".join(words)

def text_to_gloss_llm(text, api_key=None):
    api_key = api_key or os.environ.get("GROQ_API_KEY")
    if not api_key:
        return text_to_naive_gloss(text)
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        resp = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            reasoning_effort="low",
            messages=[
                {"role":"system","content":"Convert English to concise ISL gloss. Reply only with the gloss."},
                {"role":"user","content":str(text)}
            ],
        )
        return resp.choices[0].message.content.strip()
    except Exception as exc:
        print("Groq unavailable; using naive gloss:", type(exc).__name__)
        return text_to_naive_gloss(text)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# RETRIEVAL_MAX_SAMPLES is defined once in the config cell (Section 0) so it
# applies consistently across local/Kaggle runs and so the reproducibility-
# config cell can record its value regardless of execution order.
if len(df_train) > RETRIEVAL_MAX_SAMPLES:
    retrieval_df = df_train.sample(RETRIEVAL_MAX_SAMPLES, random_state=SEED)
    print(f"Using {len(retrieval_df)} samples for retrieval baseline (capped to bound memory/runtime).")
else:
    retrieval_df = df_train

train_gloss = retrieval_df["text"].map(text_to_naive_gloss).tolist()
tfidf = TfidfVectorizer().fit(train_gloss)
train_vecs = tfidf.transform(train_gloss)

def retrieve_pose_for_text(text):
    q = tfidf.transform([text_to_naive_gloss(text)])
    sims = cosine_similarity(q, train_vecs)[0]
    idx = int(np.argmax(sims))
    row = retrieval_df.iloc[idx]  # use the subset, not full df_train
    pose, length = load_pose(row, FULL_FEATURE_MASK)
    return pose[:length], row["text"], float(sims[idx])

print("Example gloss:", text_to_naive_gloss(df_test.iloc[0]["text"]))


## 12. Text → Sign: autoregressive pose generator

The generator is deliberately small enough to run on CPU. Unlike the earlier notebook, it receives a learned text context and a learned time embedding at every step.

Training uses teacher forcing; evaluation is explicitly **free-running**, so the reported result is not contaminated by ground-truth previous frames.

In [ ]:
if RUN_TEXT_TO_POSE:
    # Text-to-Pose follows the declared pipeline: English -> ISL gloss -> pose.
    # The reproducible baseline uses the deterministic naive gloss; USE_LLM_GLOSS (Section 0,
    # off by default) is the one deliberate substitution point -- flipping it swaps in the
    # Groq-backed gloss (falling back to the naive heuristic per-row on any API failure) for
    # the Text->Pose generator specifically. The retrieval baseline in Section 11 stays on the
    # naive gloss unconditionally, since it must remain a fixed comparator regardless of this
    # setting -- it should not silently change the benchmark during comparison.
    gloss_fn = text_to_gloss_llm if USE_LLM_GLOSS else text_to_naive_gloss
    if USE_LLM_GLOSS:
        print(f"USE_LLM_GLOSS=True: glossing {len(df_train) + len(df_val) + len(df_test):,} rows via "
              "Groq (falls back to the naive heuristic per-row on any API failure). This adds "
              "network latency and non-determinism relative to the naive-gloss default.")

    df_train["gloss"] = df_train["text"].map(gloss_fn)
    df_val["gloss"] = df_val["text"].map(gloss_fn)
    df_test["gloss"] = df_test["text"].map(gloss_fn)

    gloss_vocab = build_vocab(df_train["gloss"])
    gloss_inv_vocab = {i: w for w, i in gloss_vocab.items()}
    G_PAD, G_BOS, G_EOS, G_UNK = [gloss_vocab[s] for s in SPECIALS]

    def encode_gloss(text, max_len=MAX_TOKENS):
        words = str(text).lower().split()[: max_len - 2]
        return [G_BOS] + [gloss_vocab.get(w, G_UNK) for w in words] + [G_EOS]

    class TextToPoseDataset(Dataset):
        def __init__(self, frame, output_frames=POSE_OUTPUT_FRAMES):
            self.rows = frame.reset_index(drop=True)
            self.output_frames = output_frames

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, idx):
            row = self.rows.iloc[idx]
            tokens = torch.tensor(encode_gloss(row["gloss"]), dtype=torch.long)
            pose, _ = load_pose(row, FULL_FEATURE_MASK, max_frames=self.output_frames)
            return tokens, torch.from_numpy(np.ascontiguousarray(pose)).float()

    def collate_text_pose(batch):
        tokens, poses = zip(*batch)
        max_len = max(len(t) for t in tokens)
        padded = torch.full((len(tokens), max_len), G_PAD, dtype=torch.long)
        for i, t in enumerate(tokens):
            padded[i, :len(t)] = t
        return padded, torch.stack(poses)

    train_pose_ds = TextToPoseDataset(df_train)
    val_pose_ds = TextToPoseDataset(df_val)
    test_pose_ds = TextToPoseDataset(df_test)

    train_pose_loader = DataLoader(train_pose_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_text_pose, **_loader_kwargs)
    val_pose_loader = DataLoader(val_pose_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_text_pose, **_loader_kwargs)
    test_pose_loader = DataLoader(test_pose_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_text_pose, **_loader_kwargs)

    class TextToPoseGRU(nn.Module):
        def __init__(self, vocab_size, feature_dim, hidden=192, output_frames=POSE_OUTPUT_FRAMES):
            super().__init__()
            self.embed = nn.Embedding(vocab_size, hidden, padding_idx=G_PAD)
            self.context = nn.Linear(hidden, hidden)
            self.time = nn.Embedding(output_frames, hidden)
            self.cell = nn.GRUCell(feature_dim + hidden, hidden)
            self.out = nn.Linear(hidden, feature_dim)
            self.output_frames = output_frames
            self.feature_dim = feature_dim

        def encode_text(self, tokens):
            mask = (tokens != G_PAD).float().unsqueeze(-1)
            emb = self.embed(tokens) * mask
            return torch.tanh(self.context(emb.sum(1) / mask.sum(1).clamp(min=1.0)))

        def forward(self, tokens, teacher=None):
            hidden = self.encode_text(tokens)
            batch = tokens.size(0)
            prev = torch.zeros(batch, self.feature_dim, device=tokens.device)
            outputs = []
            for t in range(self.output_frames):
                ctx = self.time(torch.tensor(t, device=tokens.device)).expand(batch, -1)
                hidden = self.cell(torch.cat([prev, ctx], dim=-1), hidden)
                pred = self.out(hidden)
                outputs.append(pred)
                if teacher is not None:
                    prev = teacher[:, t]
                else:
                    prev = pred
            return torch.stack(outputs, dim=1)

    def pose_loss(pred, target):
        position = nn.functional.mse_loss(pred, target)
        velocity = nn.functional.mse_loss(pred[:, 1:] - pred[:, :-1], target[:, 1:] - target[:, :-1])
        return position + 0.25 * velocity

    pose_model = TextToPoseGRU(len(gloss_vocab), LANDMARK_DIM).to(DEVICE)
    pose_opt = torch.optim.AdamW(pose_model.parameters(), lr=1e-3, weight_decay=1e-4)

    best = float("inf")
    pose_history = []
    n_train_batches = len(train_pose_loader)
    for epoch in range(1, N_EPOCHS_POSE + 1):
        pose_model.train()
        tr_total = 0.0
        pose_opt.zero_grad(set_to_none=True)
        pbar = tqdm(train_pose_loader, desc="TextToPose train", leave=False, total=n_train_batches)
        for i, (tokens, poses) in enumerate(pbar):
            tokens, poses = tokens.to(DEVICE), poses.to(DEVICE)
            with autocast_ctx():
                pred = pose_model(tokens, teacher=poses)
                loss = pose_loss(pred, poses)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite Text-to-Pose loss.")
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            is_last = (i == n_train_batches - 1)
            if (i + 1) % GRAD_ACCUM_STEPS == 0 or is_last:
                scaler.unscale_(pose_opt)
                torch.nn.utils.clip_grad_norm_(pose_model.parameters(), 1.0)
                scaler.step(pose_opt)
                scaler.update()
                pose_opt.zero_grad(set_to_none=True)
            tr_total += loss.item()
            pbar.set_postfix(loss=f"{tr_total / (i + 1):.4f}")
            del tokens, poses, pred, loss

        pose_model.eval()
        va_total = 0.0
        with torch.no_grad():
            for tokens, poses in tqdm(val_pose_loader, desc="TextToPose eval", leave=False):
                tokens, poses = tokens.to(DEVICE), poses.to(DEVICE)
                with autocast_ctx():
                    pred = pose_model(tokens, teacher=None)
                    batch_loss = pose_loss(pred, poses)
                va_total += batch_loss.item()
                del tokens, poses, pred, batch_loss

        tr = tr_total / max(1, n_train_batches)
        va = va_total / max(1, len(val_pose_loader))
        pose_history.append((epoch, tr, va))
        print(f"[TextToPose] epoch {epoch:02d} train={tr:.5f} val={va:.5f}")

        if va < best:
            best = va
            torch.save(pose_model.state_dict(), str(CHECKPOINT_DIR / "signsage_text_to_pose_best.pt"))

        gc.collect()
        maybe_check_memory()

    pose_model.load_state_dict(torch.load(str(CHECKPOINT_DIR / "signsage_text_to_pose_best.pt"), map_location=DEVICE))
else:
    gloss_fn=None
    pose_model=None; pose_history=[]
    retrieval_dtw=[]; retrieval_mse=[]; generative_dtw=[]; generative_mse=[]
    gt_motion=[]; retrieval_motion=[]; generative_motion=[]
    t2p_metrics={}
    print("Text→Pose branch skipped. It remains available as an optional M.Tech research experiment.")


In [ ]:
if RUN_TEXT_TO_POSE:
    # 12a. Record Text→Pose experiment history and checkpoint.
    pose_ckpt = CHECKPOINT_DIR / "signsage_text_to_pose_best.pt"
    pose_best_val = float(min(v for _, _, v in pose_history)) if pose_history else float("nan")
    knowledge_insert(
        "experiments",
        ["run_id", "name", "family", "input_features", "epochs", "batch_size",
         "learning_rate", "weight_decay", "checkpoint_path", "best_validation_loss"],
        [RUN_ID, "Text→Pose GRU", "Text→Sign", "ISL gloss → pose", N_EPOCHS_POSE, BATCH_SIZE,
         1e-3, 1e-4, str(pose_ckpt), pose_best_val],
        delete_keys={"run_id": RUN_ID, "name": "Text→Pose GRU"},
    )
    for epoch, tr_loss, val_loss in pose_history:
        knowledge_metric("Text→Pose GRU", f"train_loss_epoch_{epoch}", tr_loss, "train")
        knowledge_metric("Text→Pose GRU", f"val_loss_epoch_{epoch}", val_loss, "validation")
    knowledge_artifact("text_to_pose_gru", "checkpoint", pose_ckpt, "Best Text→Pose GRU checkpoint.")
    print("Text→Pose experiment recorded.")
else:
    print("Text→Pose experiment record skipped.")


## 13. Text → Sign evaluation

Retrieval and generation are compared on the same held-out test examples.

DTW is computed on normalized pose coordinates, and a simple normalized framewise MSE is also reported. Neither metric alone establishes sign-language quality; the static visualization and round-trip check are additional sanity checks.

In [ ]:
if RUN_TEXT_TO_POSE:
    def dtw_distance(a, b):
        # Memory-safe O(nm) DP for short evaluation sequences.
        a = np.asarray(a, dtype=np.float32)
        b = np.asarray(b, dtype=np.float32)
        n, m = len(a), len(b)
        prev = np.full(m + 1, np.inf, dtype=np.float64)
        prev[0] = 0.0
        for i in range(1, n + 1):
            cur = np.full(m + 1, np.inf, dtype=np.float64)
            d = np.linalg.norm(a[i-1][None, :] - b, axis=1)
            cur[1:] = d + np.minimum(np.minimum(prev[1:], cur[:-1]), prev[:-1])
            prev = cur
        return float(prev[m])

    @torch.no_grad()
    def generate_pose(text):
        # BUG FIX: must use gloss_fn (Section 12), the same gloss function gloss_vocab and
        # pose_model were trained on. Hardcoding text_to_naive_gloss here meant that turning on
        # USE_LLM_GLOSS would silently break generation: tokens would be looked up against a
        # vocabulary built from a different gloss function and drift toward <unk>.
        tokens = torch.tensor([encode_gloss(gloss_fn(text))], dtype=torch.long, device=DEVICE)
        return pose_model(tokens, teacher=None)[0].cpu().numpy()

    def motion_energy(pose_seq):
        """Mean absolute frame-to-frame change -- a cheap proxy for 'how much does this
        sequence actually move'. A model that collapses to predicting a near-static/average
        pose can still score well on DTW/MSE against ground truth (regression-to-the-mean),
        so this is checked as an explicit, separate safeguard rather than trusting DTW/MSE
        alone to catch that failure mode."""
        if len(pose_seq) < 2:
            return 0.0
        diffs = pose_seq[1:] - pose_seq[:-1]
        return float(np.mean(np.abs(diffs)))

    eval_n = min(DTW_EVAL_N, len(df_test))
    retrieval_dtw, generative_dtw = [], []
    retrieval_mse, generative_mse = [], []
    gt_motion, retrieval_motion, generative_motion = [], [], []

    for _, row in df_test.iloc[:eval_n].iterrows():
        gt, _ = load_pose(row, FULL_FEATURE_MASK, max_frames=POSE_OUTPUT_FRAMES)
        gt = gt[:POSE_OUTPUT_FRAMES]

        retrieved, _, _ = retrieve_pose_for_text(row["text"])
        retrieved = resize_sequence(retrieved, POSE_OUTPUT_FRAMES)

        generated = generate_pose(row["text"])

        retrieval_dtw.append(dtw_distance(retrieved, gt))
        generative_dtw.append(dtw_distance(generated, gt))
        retrieval_mse.append(float(np.mean((retrieved - gt) ** 2)))
        generative_mse.append(float(np.mean((generated - gt) ** 2)))

        gt_motion.append(motion_energy(gt))
        retrieval_motion.append(motion_energy(retrieved))
        generative_motion.append(motion_energy(generated))

    print(f"Retrieval : DTW mean={np.mean(retrieval_dtw):.3f} ± {np.std(retrieval_dtw):.3f}; MSE={np.mean(retrieval_mse):.5f}")
    print(f"Generative: DTW mean={np.mean(generative_dtw):.3f} ± {np.std(generative_dtw):.3f}; MSE={np.mean(generative_mse):.5f}")

    print(f"\nMotion energy (mean |frame-to-frame change|) over {eval_n} test examples:")
    print(f"  Ground truth : {np.mean(gt_motion):.5f}")
    print(f"  Retrieval    : {np.mean(retrieval_motion):.5f}")
    print(f"  Generative   : {np.mean(generative_motion):.5f}")

    _gen_motion_ratio = (np.mean(generative_motion) / np.mean(gt_motion)) if np.mean(gt_motion) > 0 else float("nan")
    print(f"  Generative/Ground-truth motion ratio: {_gen_motion_ratio:.3f}")
    if np.isfinite(_gen_motion_ratio) and _gen_motion_ratio < 0.3:
        print("  WARNING: generated poses move far less than ground truth -- this suggests the\n"
              "  generator may have collapsed to a near-static/average pose rather than producing\n"
              "  real motion. A good DTW/MSE score alongside a low motion ratio should NOT be read\n"
              "  as evidence of successful pose generation on its own; inspect the visualization in\n"
              "  the next section before treating this as a positive result.")

    knowledge_observation("text_to_pose_motion_energy", {
        "eval_n": int(eval_n),
        "ground_truth_motion": float(np.mean(gt_motion)),
        "retrieval_motion": float(np.mean(retrieval_motion)),
        "generative_motion": float(np.mean(generative_motion)),
        "generative_to_gt_motion_ratio": float(_gen_motion_ratio) if np.isfinite(_gen_motion_ratio) else None,
    })
else:
    print("Text→Pose evaluation skipped.")


In [ ]:
if RUN_TEXT_TO_POSE:
    # 13a. Record Text→Pose evaluation metrics.
    t2p_metrics = {
        "retrieval_DTW_mean": float(np.mean(retrieval_dtw)),
        "retrieval_DTW_std": float(np.std(retrieval_dtw)),
        "retrieval_MSE_mean": float(np.mean(retrieval_mse)),
        "generative_DTW_mean": float(np.mean(generative_dtw)),
        "generative_DTW_std": float(np.std(generative_dtw)),
        "generative_MSE_mean": float(np.mean(generative_mse)),
        "ground_truth_motion_energy": float(np.mean(gt_motion)),
        "retrieval_motion_energy": float(np.mean(retrieval_motion)),
        "generative_motion_energy": float(np.mean(generative_motion)),
    }
    for metric_name, metric_value in t2p_metrics.items():
        knowledge_metric("Text→Pose", metric_name, metric_value, "test")
    knowledge_observation("text_to_pose_metrics", t2p_metrics)
    print("Text→Pose evaluation metrics recorded.")
else:
    print("Text→Pose metrics skipped.")


In [ ]:
if RUN_TEXT_TO_POSE:
    # Qualitative Text -> Sign examples.
    for _, row in df_test.head(5).iterrows():
        retrieved, matched, sim = retrieve_pose_for_text(row["text"])
        generated = generate_pose(row["text"])
        print("TEXT      :", row["text"])
        print("GLOSS     :", row["gloss"], "(LLM)" if USE_LLM_GLOSS else "(naive)")
        print("RETRIEVED :", matched, f"(sim={sim:.3f}, frames={len(retrieved)})")
        print("GENERATED :", generated.shape)
        print()
else:
    print("Text→Pose qualitative examples skipped.")


## 14. Pose visualization

In [ ]:
if RUN_TEXT_TO_POSE:
    import matplotlib.pyplot as plt

    sample_text = df_test.iloc[0]["text"]
    generated = generate_pose(sample_text)

    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    for ax, fi in zip(axes, np.linspace(0, len(generated)-1, 6).astype(int)):
        pts = generated[fi].reshape(543, 3)
        # Show pose + hands; face is omitted from the static plot for readability.
        visible = np.concatenate([pts[:33], pts[501:543]], axis=0)
        ax.scatter(visible[:,0], -visible[:,1], s=7)
        ax.set_title(f"frame {fi}")
        ax.axis("equal")
        ax.axis("off")
    fig.suptitle(f'Text → generated pose: "{sample_text}"')
    plt.tight_layout()
    plt.show()
else:
    print("Pose visualization skipped in primary Sign→English run.")


## 15. Real round-trip sanity check

The previous notebook's round-trip cell was conceptually incomplete: it generated a pose, but then fed the **ground-truth pose from the original dataset row** into Sign → Text.

This cell fixes that. It generates a pose from text, then feeds the **generated pose** into the Sign → Text Transformer.

This is a model-level round trip, not a claim of live webcam performance.

In [ ]:
if RUN_TEXT_TO_POSE:
    @torch.no_grad()
    def sign_text_from_pose_array(pose_array):
        # pose_array is normalized 543-landmark data.
        feature = pose_array[:, FULL_FEATURE_MASK]
        true_len = min(len(feature), MAX_FRAMES)
        feature = resize_sequence(feature, MAX_FRAMES)
        frames = torch.from_numpy(np.ascontiguousarray(feature)).float()
        return constrained_greedy_transformer(transformer_full, frames, true_len)

    roundtrip_text = df_test.iloc[0]["text"]
    roundtrip_gloss = text_to_naive_gloss(roundtrip_text)
    roundtrip_pose = generate_pose(roundtrip_text)
    roundtrip_recovered = sign_text_from_pose_array(roundtrip_pose)

    print("Original text :", roundtrip_text)
    print("Gloss         :", roundtrip_gloss)
    print("Generated pose:", roundtrip_pose.shape)
    print("Recovered text:", roundtrip_recovered)
else:
    roundtrip_text=None; roundtrip_gloss=None; roundtrip_pose=None; roundtrip_recovered=None
    print("Text→Pose round-trip skipped. Primary validation is direct held-out Sign→English inference.")


## 15. Live inference contract

The notebook's model input is **a sequence of 543 MediaPipe-compatible landmarks**, not a single webcam frame.
The web backend must collect a motion segment, apply the exact normalization below, then call this function.
No raw video is required by the trained model.

In [ ]:
@torch.inference_mode()
def prepare_live_landmarks(landmarks_xyz):
    """Convert T×543×3 camera landmarks into the exact model representation."""
    arr=np.asarray(landmarks_xyz,dtype=np.float32)
    if arr.ndim!=3 or arr.shape[1:]!=(543,3):
        raise ValueError(f"Expected (T,543,3), received {arr.shape}")
    if len(arr)<2:
        raise ValueError("Need at least two frames.")
    flat=arr.reshape(len(arr),-1)
    norm=normalize_pose_sequence(flat)
    true_len=min(len(norm),MAX_FRAMES)
    return resize_sequence(norm,MAX_FRAMES),true_len

def live_motion_energy(landmarks_xyz):
    arr=np.asarray(landmarks_xyz,dtype=np.float32)
    if len(arr)<2:return 0.0
    # Activity gate is computed on root-centered, normalized landmarks so camera
    # translation/scale does not dominate it.
    norm=normalize_pose_sequence(arr.reshape(len(arr),-1))
    return float(np.mean(np.abs(np.diff(norm,axis=0))))

def predict_live_landmarks(landmarks_xyz, min_motion=0.002, min_confidence=None):
    processed,true_len=prepare_live_landmarks(landmarks_xyz)
    motion=live_motion_energy(landmarks_xyz)
    text,confidence=constrained_greedy_transformer(
        transformer_full,torch.from_numpy(processed).float(),true_len,return_confidence=True)
    if min_confidence is None:
        min_confidence=float(globals().get("LIVE_CONFIDENCE_FLOOR",0.0))
    accepted=bool(motion>=min_motion and confidence>=min_confidence and normalize_text(text))
    return {
        "text": text if accepted else "",
        "accepted": accepted,
        "motion_energy": motion,
        "confidence_proxy": confidence,
        "reason": "ok" if accepted else ("insufficient_motion" if motion<min_motion else "low_model_confidence"),
    }

print("Live inference contract defined: MediaPipe landmarks → shared normalization → Transformer → English.")


In [ ]:
# 15a. Training/inference contract self-test
# This verifies that the exact same landmark dimensionality and normalization
# path used by the model can consume a real held-out .pose sequence.
sample_frames, sample_len, sample_tokens = test_ds_full[0]
assert sample_frames.shape == (MAX_FRAMES, LANDMARK_DIM)
assert 1 <= sample_len <= MAX_FRAMES

# Convert the normalized training tensor back only for a shape/contract check.
# The actual backend will provide raw MediaPipe T×543×3 landmarks and call
# prepare_live_landmarks(), which is the canonical live preprocessing entrypoint.
contract_input = np.zeros((max(2, sample_len), 543, 3), dtype=np.float32)
contract_processed, contract_len = prepare_live_landmarks(contract_input)
assert contract_processed.shape == (MAX_FRAMES, LANDMARK_DIM)
assert contract_len == min(max(2, sample_len), MAX_FRAMES)

# A constant zero sequence must never produce a translated sentence.
zero_result = predict_live_landmarks(contract_input, min_motion=0.002, min_confidence=0.0)
assert not zero_result["accepted"], "Activity gate failed: constant landmarks were accepted."
print("Live inference contract self-test passed.")
print("Constant/no-motion input correctly abstains from translation.")


## 16. Latency harness

These are **model-only** timings. They are not end-to-end browser → WebSocket → backend → model timings and must not be presented as the project's final system latency.

In [ ]:
## 16. Latency harness
def measure_ms(fn,*args,runs=5):
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    vals=[]
    for _ in range(runs):
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        t0=time.perf_counter(); fn(*args)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        vals.append((time.perf_counter()-t0)*1000)
    return float(np.mean(vals)),float(np.std(vals))

sample_frames,sample_len,_=test_ds_full[0]
s2t_mean,s2t_std=measure_ms(constrained_greedy_transformer,transformer_full,sample_frames,sample_len)
t2p_mean=t2p_std=float("nan")
print(f"Sign→Text model inference: {s2t_mean:.2f} ± {s2t_std:.2f} ms")
print("This excludes MediaPipe, WebSocket, network and frontend rendering.")


In [ ]:
# 16a. Record runtime and deployment metadata.
knowledge_metric("Runtime","sign_to_text_latency_ms_mean",s2t_mean,"inference")
knowledge_metric("Runtime","sign_to_text_latency_ms_std",s2t_std,"inference")
knowledge_observation("deployment_contract",{
    "input":"normalized 543-landmark sequence",
    "max_frames":MAX_FRAMES,
    "feature_dim":LANDMARK_DIM,
    "output":"English text",
    "decoder":"constrained greedy for runtime; beam only for qualitative analysis",
    "raw_video_stored":False,
})


## 17. Automatic go / no-go report

This section does not pretend a failing experiment is a success. It tells you exactly what the current run established.

**Interpretation rules:**

- GRU training loss should decrease without exploding while validation remains finite.
- Transformer should beat the trivial baseline on at least the primary translation metric(s) before being described as successful.
- Face ablation is interpretable because the mask is verified from the actual pose header.
- Text → Pose generation should be compared against retrieval rather than judged by MSE alone.
- The round-trip result is qualitative evidence only; it is not an independent benchmark.

In [ ]:
## 17. Automatic GO / NO-GO report for the actual objective

primary_metrics=results_s2t["Transformer (full)"]
transformer_beats_trivial=(
    primary_metrics["WER"] < results_s2t["Trivial baseline"]["WER"]
    or primary_metrics["BLEU"] > results_s2t["Trivial baseline"]["BLEU"]
)
translation_not_collapsed=(tf_unique_frac >= 0.30)

print("=== SignSage Sign→English status ===")
print("Full usable training rows:", f"{len(df_train):,}")
print("Full usable validation rows:", f"{len(df_val):,}")
print("Full usable test rows:", f"{len(df_test):,}")
print("Video-level leakage-free:", leakage_free)
print("Transformer beats trivial baseline:", transformer_beats_trivial)
print("Output diversity >= 30%:", translation_not_collapsed)
print("Test WER:", round(primary_metrics["WER"],4))
print("Test BLEU:", round(primary_metrics["BLEU"],2))
print("Test ROUGE-L:", round(primary_metrics["ROUGE-L"],4))

if transformer_beats_trivial and translation_not_collapsed:
    print("\nRESULT: model shows measurable translation signal without obvious near-fixed-output collapse.")
else:
    print("\nRESULT: NO-GO for deployment. Do not connect this checkpoint to the webcam backend yet.")
    if not transformer_beats_trivial: print(" - Translation metrics do not beat the trivial baseline.")
    if not translation_not_collapsed: print(" - Output diversity is too low; inspect data/model behavior.")

knowledge_observation("go_no_go",{
    "transformer_beats_trivial":bool(transformer_beats_trivial),
    "output_diversity_ok":bool(translation_not_collapsed),
    "test_metrics":primary_metrics,
    "phase_0_status":"GO-CANDIDATE" if transformer_beats_trivial and translation_not_collapsed else "NO-GO",
})


In [ ]:
# 18. Finalize experiment + deployment artifacts

# The backend must never need the training notebook itself. Export the exact
# vocabulary/configuration used by the primary Sign→English model.
deployment_config={
    "project":"SignSage",
    "task":"ISL_to_English",
    "dataset":"iSign v1.1",
    "feature_dim":int(LANDMARK_DIM),
    "model_input_dim":int(LANDMARK_DIM*2),
    "motion_features":"per-frame delta of normalized landmarks; first frame zero",
    "landmarks":543,
    "coordinates":3,
    "max_frames":int(MAX_FRAMES),
    "max_tokens":int(MAX_TOKENS),
    "vocab_size":int(len(vocab)),
    "tokenizer":"SentencePiece unigram trained on training targets only",
    "tokenizer_model":"tokenizer/signsage_en_sp.model",
    "special_tokens":SPECIALS,
    "normalization":"root-center on body hips (23,24), divide by per-sequence body landmark std; NaN/Inf -> 0",
    "decoder_runtime":"constrained_greedy",
    "beam_width_qualitative":int(BEAM_WIDTH),
    "repetition_penalty":float(REPETITION_PENALTY),
    "no_repeat_ngram_size":int(NO_REPEAT_NGRAM_SIZE),
    "raw_video_required_by_model":False,
    "training_rows":int(len(df_train)),
    "validation_rows":int(len(df_val)),
    "test_rows":int(len(df_test)),
    "video_level_split":True,
    "checkpoint":"signsage_transformer_full.pt",
}
with open(REPORT_DIR/"sign_to_text_deployment.json","w",encoding="utf-8") as f:
    json.dump(deployment_config,f,indent=2,ensure_ascii=False)
with open(REPORT_DIR/"token_vocabulary.json","w",encoding="utf-8") as f:
    json.dump({"token_to_id":vocab,"special_tokens":SPECIALS,"tokenizer_type":"sentencepiece_unigram","tokenizer_model":str(TOKENIZER_MODEL)},f,indent=2,ensure_ascii=False)

knowledge_artifact("sign_to_text_deployment","json",REPORT_DIR/"sign_to_text_deployment.json",
                   "Exact Sign→English preprocessing/model/decoder contract for backend integration.")
knowledge_artifact("tokenizer_model","model",TOKENIZER_MODEL,"SentencePiece model used for English target tokenization.")
knowledge_artifact("token_vocabulary","json",REPORT_DIR/"token_vocabulary.json",
                   "Vocabulary used by the primary Sign→English checkpoint.")
knowledge_artifact("transformer_checkpoint","pt",CHECKPOINT_DIR/"signsage_transformer_full.pt",
                   "Primary full-feature Sign→English Transformer checkpoint.")

manifest={
    "project":"SignSage","repository_version":16,"run_id":RUN_ID,
    "started_at":RUN_STARTED_AT,"completed_at":pd.Timestamp.utcnow().isoformat(),
    "task":"ISL_to_English",
    "environment":{"data_mode":DATA_MODE,"run_mode":RUN_MODE,"device":str(DEVICE),
                   "gpu_count":int(torch.cuda.device_count()),"seed":SEED},
    "tokenizer":{"type":"sentencepiece_unigram","vocab_size":int(sp.get_piece_size()),"model":str(TOKENIZER_MODEL)},
    "dataset":{"name":"iSign v1.1","csv_rows":int(len(df)),
               "unique_uids":int(df.uid.nunique()),"unique_video_ids":int(df.video_id.nunique()),
               "selected_rows":int(len(df_train)+len(df_val)+len(df_test))},
    "split":{"train_rows":int(len(df_train)),"val_rows":int(len(df_val)),
             "test_rows":int(len(df_test)),"leakage_free":bool(leakage_free)},
    "configuration":deployment_config,
    "evaluation":{"sign_to_text":results_s2t,
                  "output_diversity_fraction":float(tf_unique_frac),
                  "mean_confidence_proxy":float(np.mean(tf_conf))},
    "status":{"transformer_beats_trivial":bool(transformer_beats_trivial),
              "phase_0_status":"GO-CANDIDATE" if transformer_beats_trivial and translation_not_collapsed else "NO-GO"},
}
with open(KNOWLEDGE_JSON_PATH,"w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2,ensure_ascii=False)

conn=_sql_connect()
conn.execute("UPDATE runs SET completed_at=? WHERE run_id=?",(manifest["completed_at"],RUN_ID))
conn.commit(); conn.close()
print("Experiment repository finalized.")
print("Deployment config:",REPORT_DIR/"sign_to_text_deployment.json")
print("Checkpoint:",CHECKPOINT_DIR/"signsage_transformer_full.pt")


## 19. Kaggle → SignSage hand-off

For the primary application, copy these artifacts from `signsage_outputs/`:

signsage_outputs/
├── checkpoints/
│   └── signsage_transformer_full.pt
├── reports/
│   ├── token_vocabulary.json
│   └── sign_to_text_deployment.json
├── signsage_knowledge.db
└── signsage_knowledge_manifest.json

The web backend should load the checkpoint once at startup and perform inference only.
It must reuse the exported normalization, feature ordering, vocabulary and frame limit.

IMPORTANT:
- The notebook trains the model; it is not executed for every webcam request.
- The backend should not send raw video to the model.
- If the webcam window is below the activity/confidence gate, return "no sign detected"
  rather than fabricating English.
- Final live webcam behavior is validated only after the backend uses this exact contract.


In [ ]:
# 20. Final artifact inventory
artifact_paths = sorted([
    p for p in OUTPUT_DIR.rglob("*")
    if p.is_file()
])

print(f"Artifacts written: {len(artifact_paths)}")
for p in artifact_paths:
    size_mb = p.stat().st_size / (1024**2)
    print(f"{p.relative_to(OUTPUT_DIR)} — {size_mb:.2f} MB")

print("Notebook revision: FULL-CORPUS MEMORY-SAFE DEADLINE FINAL — complete cleaned iSign corpus, optimized remote loading, and deployment-bundle export.")


In [ ]:
# 19. Create a compact deployment bundle for the web application
#
# This is the ONLY output bundle needed later for backend integration.
# It deliberately excludes dataset files, cache files, training logs, and
# per-example predictions. The large Kaggle output ZIP is therefore unnecessary.
import shutil, zipfile, json
BUNDLE_DIR = OUTPUT_DIR / "signsage_deployment_bundle"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

required_artifacts = [
    (CHECKPOINT_DIR / "signsage_transformer_full.pt", "signsage_transformer_full.pt"),
    (REPORT_DIR / "tokenizer" / "signsage_en_sp.model", "tokenizer/signsage_en_sp.model"),
    (REPORT_DIR / "sign_to_text_deployment.json", "sign_to_text_deployment.json"),
    (REPORT_DIR / "token_vocabulary.json", "token_vocabulary.json"),
    (KNOWLEDGE_JSON_PATH, "signsage_knowledge_manifest.json"),
]
missing_artifacts = [str(src) for src, _ in required_artifacts if not src.exists()]
if missing_artifacts:
    raise RuntimeError("Deployment bundle is missing required artifacts: " + ", ".join(missing_artifacts))

for src, rel in required_artifacts:
    dst = BUNDLE_DIR / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

integration_note = BUNDLE_DIR / "INTEGRATION_NOTE.txt"
integration_note.write_text(
    "SignSage Sign→Text deployment bundle.\n"
    "Use the checkpoint together with the SentencePiece model and "
    "sign_to_text_deployment.json.\n"
    "The backend receives RAW 1629-D landmark frames (543 landmarks × 3 coordinates).\n"
    "The model internally concatenates each frame with its frame-to-frame delta, "
    "so the model input is 3258-D.\n"
    "Raw camera/video is not required by this model.\n"
    "Do not upload the iSign dataset or training cache to the web application.\n",
    encoding="utf-8",
)

bundle_zip = OUTPUT_DIR / "signsage_deployment_bundle.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in BUNDLE_DIR.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(BUNDLE_DIR))

bundle_size_mb = bundle_zip.stat().st_size / (1024**2)
print("Compact deployment bundle:", bundle_zip)
print(f"Bundle size: {bundle_size_mb:.1f} MB")
print("Contents:")
for p in sorted(BUNDLE_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(BUNDLE_DIR))
print("\nDo NOT use the full Kaggle output ZIP for web-app integration; use this bundle.")
